# Polaris API -> OpenSearch: every operation, every reachable status

**What this asks.** v1 (`polaris_log_coverage.ipynb`) asks *"if something went wrong on this
endpoint tomorrow, would there be a record of it?"* and answers it for 40 of the 63 operations,
at whatever status those calls happened to return. This notebook widens both axes: **every
operation the vendored specs name, at every status this build can actually produce**, verified
against the **OpenSearch** side of the pipeline with the gates of `GUIDE-schema-v3-testing.md`.

Plan: [`PLAN-api-status-matrix.md`](PLAN-api-status-matrix.md). It is the read, and it carries
the corrections its own predictions earned.

## Before you run this

```bash
kubectl -n <os-ns> port-forward svc/<opensearch> 9200:9200     # or reach it directly
export OPENSEARCH_PASS=...                                     # never in a config file
```

`local` env only, and it **mutates**: a catalog, principals, roles, namespaces, a table and a
view are created and deleted. ~300 calls.

## The two things that make this different from v1

**The sink is OpenSearch**, not VictoriaLogs, and they are fed by *different* Fluent Bits
running *different* schema versions (measured 2026-09-10: DaemonSet v3 -> OpenSearch,
Deployment v2 -> VictoriaLogs). Nothing verified on one carries to the other.

**The run is phased on window boundaries.** Every gate in the guide is per-window, so phases E,
F and G each get a window to themselves -- `last_write_bytes` means "the last 2xx POST in this
window" and that is only unambiguous when there is one.

## 0 -- Preflight

Nothing below is diagnosed until this passes. A check that cannot run says so; none of them
print a tidy zero.

In [1]:
import pathlib
import shutil
import subprocess
import sys
import time
from datetime import datetime, timezone

_SRC = pathlib.Path.cwd().parent / "src"
if str(_SRC) not in sys.path:
    sys.path.insert(0, str(_SRC))

from polaris_test_utils import *  # noqa: F401,F403
import polaris_test_utils as ptu
import api_status_matrix as mx
import log_coverage as lc
import os_report as osr

init_env("local")
require_not_prod("the api-status-matrix drive creates and deletes catalogs, principals, roles, "
                 "namespaces, a table and a view, and provokes 500s")

RUN = str(int(time.time()))
STARTED = datetime.now(timezone.utc)
PRE, FAIL = {}, []


def sh(*args, timeout=25):
    """kubectl, when it is there. Returns stdout or None -- never raises."""
    if not shutil.which(args[0]):
        return None
    try:
        r = subprocess.run(args, capture_output=True, text=True, timeout=timeout)
        return r.stdout.strip() if r.returncode == 0 else None
    except Exception:
        return None


OS_URL = f"https://{ptu.OPENSEARCH_HOST}:{ptu.OPENSEARCH_PORT}"
OS = osr.OSReports(
    OS_URL,
    auth=(ptu.OPENSEARCH_USER, ptu.OPENSEARCH_PASS),
    report_index=ptu.CFG.get("opensearch_report_index", osr.DEFAULT_REPORT_INDEX),
    logs_index=ptu.CFG.get("opensearch_logs_index", osr.DEFAULT_LOGS_INDEX),
)

try:
    ROOT_TOK = root_token()
    PRE["polaris"] = f"OK   {ptu.POLARIS_URL}"
except Exception as exc:
    ROOT_TOK = None
    FAIL.append(f"Polaris: {type(exc).__name__}: {exc}")

ok, detail = OS.ping()
PRE["opensearch"] = f"{'OK  ' if ok else 'DOWN'} {OS_URL}  ({detail})"
if not ok:
    FAIL.append(f"OpenSearch unreachable at {OS_URL} -- port-forward it, and check OPENSEARCH_PASS")

# DO THE INDICES EXIST AT ALL? README.md describes the OpenSearch shipper as a
# DIFFERENT Fluent Bit writing k8s-logs-*, so polaris-report-* / polaris-logs-*
# existing is a fact about THIS deployment and not an assumption the guide gets
# to make on our behalf.
IDX = {}
if ok:
    for row in OS.indices("polaris-*"):
        IDX[row["index"]] = row.get("docs.count")
    PRE["indices"] = ", ".join(f"{k} ({v})" for k, v in sorted(IDX.items())) or "NONE"
    if not any(k.startswith("polaris-report") for k in IDX):
        FAIL.append("no polaris-report-* index: gates 0-6 have no source. Nothing below would "
                    "fail -- it would all report zero, which is worse.")
    if not any(k.startswith("polaris-logs") for k in IDX):
        FAIL.append("no polaris-logs-* index: the per-call half of the matrix has no source.")

for k, v in PRE.items():
    print(f"  {k:22} {v}")
if FAIL:
    raise RuntimeError("PREFLIGHT FAILED:\n  - " + "\n  - ".join(FAIL))
print(f"\n  run={RUN}  started={STARTED.isoformat()}")

🟢 LOCAL   POLARIS_ENV=local   (Polaris 1.3.0)
   Polaris: http://192.168.139.2:8181
   MinIO:   http://192.168.139.2:9000  bucket=data-catalog-bucket
   OpenSearch: localhost:9200  index=k8s-logs-*
   purge_deletes_files=False
🟢 LOCAL   POLARIS_ENV=local   (Polaris 1.3.0)
   Polaris: http://192.168.139.2:8181
   MinIO:   http://192.168.139.2:9000  bucket=data-catalog-bucket
   OpenSearch: localhost:9200  index=k8s-logs-*
   purge_deletes_files=False
  polaris                OK   http://192.168.139.2:8181
  opensearch             OK   https://localhost:9200  (HTTP 200 opensearch 3.5.0)
  indices                polaris-logs-2026.09.09 (61290), polaris-logs-2026.09.10 (20471), polaris-logs-2026.09.14 (16538), polaris-report-2026.09.09 (1461), polaris-report-2026.09.10 (1881), polaris-report-2026.09.14 (937)

  run=1789370776  started=2026-09-14T07:26:16.071528+00:00


## 0a -- Gate 7 FIRST: the Lua unit test

The guide is explicit that this runs **before** any query, and the reason is worth restating: it
separates *"the filter is wrong"* from *"my query is wrong"* in one step, and it has already
caught an invalid Lua escape that would have failed the whole chunk at load.

It needs the `local-k8s` checkout and a `lua5.4`. Both are on the machine running Jupyter and
neither is a reason to skip silently -- if it cannot run, that is recorded as **UNPROVEN** and
carried into the findings.

⚠ The two documents disagree about how many assertions this harness has: `SCHEMA-report.md` says
16, `GUIDE-schema-v3-testing.md` says 46. Whatever it prints is recorded verbatim; the count is
not asserted on.

**A FAIL here is classified, not fatal.** Failures are matched against `GATE7_SCOPED` -- a list
of defects already measured and already written into `REPORT-for-local-k8s.md`, currently the
idle-window `min/max_record_time` bug. If every failure is on that list the gate records
**FAIL (scoped)**, the defect is carried into the findings as `local-k8s` work, and the run
continues: gates 1-6 read `requests`, `writes`, `last_write_bytes` and `auth_denied`, and these
failures touch none of them. Anything off the list still stops the run -- and so does a failure
count that cannot be reconciled with the visible `[FAIL]` lines, because a failure you cannot
read is a failure you cannot classify.

This cell used to `raise` on any failure at all. On 2026-09-10 that stopped run `1789029836`
here, over those two assertions, and **cells 5-42 of this notebook have never executed.**

In [2]:
import json
import re

GATE7 = {"ran": False, "ok": None, "detail": "", "assertions": None, "source": None,
         "passes": None, "failures": [], "reported_failures": None,
         "countable": None, "scoped": None, "unscoped": []}

LUA_TEST = pathlib.Path.home() / "hynix" / "local-k8s" / "logging" / "scripts" / "test-schema-v3.lua"
LUA_BIN = shutil.which("lua5.4") or shutil.which("lua")
EXTRACTED = pathlib.Path("/tmp/polaris.lua")

# THE FAILURES THAT DO NOT BEAR ON ANY GATE BELOW.
#
# This cell used to `raise` on ANY Gate 7 failure, and on 2026-09-10 that cost
# the whole notebook: run 1789029836 stopped at this cell over two assertions
# about idle-window timestamps, and cells 5-42 have never executed in this file
# at all. The abort was disproportionate in a specific, checkable way -- gates
# 1-6 read `requests`, `writes`, `last_write_bytes` and `auth_denied`, and the
# failing assertions touch none of them.
#
# So a failure is now CLASSIFIED. Every failure on this list: record FAIL
# (scoped), carry it into the findings as `local-k8s` work, and keep running.
# Anything off this list still stops the run, because an unrecognised failure
# is exactly the case where "the filter is wrong" and "my query is wrong" have
# not been separated -- which is what this gate exists to do.
#
# This is a list of DEFECTS ALREADY MEASURED AND RECORDED, not an exemption.
# Nothing goes on it that has not been written into REPORT-for-local-k8s.md.
#
# ALL THREE ENTRIES ARE ONE CAUSE, and it is bigger than any of them: the
# deployed script reports SCHEMA_VERSION=3 and does not implement all of v3.
# `SCHEMA-report.md` specifies a `transaction` kind (added 2026-09-10) and
# min/max_record_time ABSENT on an idle window; the running text has neither,
# and `grep -c transaction` against it returns 0. Cell 6 compares the CONSTANT
# and passes, so Gate 7 is the only instrument here that can see this.
# The run therefore continues WITH A PROVENANCE STAMP (below) on every document
# it writes -- a run against a filter failing its own schema test is valid only
# while its outputs say so.
GATE7_SCOPED = (
    # EMPTY, AND THAT IS THE POINT. It held three entries on 2026-09-14 --
    # min/max_record_time and `transactions/commit` -- and run 1789365275
    # matched NONE of them: the filter was rolled (35632 bytes, pod 05:54:08Z)
    # and Gate 7 passed 58/58. Entries are retired the moment the defect is
    # fixed, because a scoped list nobody empties becomes a list of things
    # nobody looks at. The mechanism stays: it is what let the run proceed
    # while those defects were real, and it is what will raise on the next
    # failure nobody has seen yet.
)


def _scope(line):
    """The recorded reason this failure is known, or None if it is new."""
    for pattern, why in GATE7_SCOPED:
        if re.search(pattern, line):
            return why
    return None


# THE PREREQUISITE THE GUIDE STATES AND THE FIRST VERSION OF THIS CELL DROPPED.
# `test-schema-v3.lua` runs the DEPLOYED SCRIPT TEXT -- that is the whole point
# of it -- and it reads that text from /tmp/polaris.lua. Run without the
# extraction it tests whatever happens to be at that path, or nothing at all,
# and reports a failure that reads "the filter is broken" when it means "you
# did not give me a filter". That is exactly what happened on the first run.
#
# The guide extracts from `fluent-bit/values.yaml`. This takes it from the
# RUNNING ConfigMap instead, for the reason the guide gives one line later:
# reading the file proves nothing about what is running. The key is located by
# CONTENT rather than by name, because charts name it differently and a rename
# would otherwise hand the test an empty script.
def _deployed_lua():
    raw = sh("kubectl", "-n", "datahub-hynix", "get", "cm",
             "-l", "app.kubernetes.io/instance=benchmarks-fluent-bit", "-o", "json")
    if not raw:
        return None, "kubectl unavailable, or no ConfigMap for benchmarks-fluent-bit"
    try:
        doc = json.loads(raw)
    except ValueError as exc:
        return None, f"the ConfigMap JSON did not parse: {exc}"
    best = None
    for item in doc.get("items", []):
        for key, value in (item.get("data") or {}).items():
            if not isinstance(value, str):
                continue
            if "polaris_noise_filter" in value or "build_report" in value:
                if best is None or len(value) > len(best[1]):
                    best = (key, value)
    if best is None:
        return None, "no ConfigMap key contained polaris_noise_filter or build_report"
    return best, ""


_pair, _why = _deployed_lua()
if _pair:
    EXTRACTED.write_text(_pair[1], encoding="utf-8")
    GATE7["source"] = f"ConfigMap key {_pair[0]} ({len(_pair[1])} bytes) -> {EXTRACTED}"
    print(f"  extracted {GATE7['source']}")
else:
    GATE7["detail"] = f"could not extract the deployed Lua: {_why}"

if not LUA_BIN:
    GATE7["detail"] = "no lua5.4 on PATH (brew install lua) -- UNPROVEN, not passed"
elif not LUA_TEST.exists():
    GATE7["detail"] = f"{LUA_TEST} not found -- UNPROVEN, not passed"
elif not _pair:
    pass
else:
    r = subprocess.run([LUA_BIN, str(LUA_TEST)], capture_output=True, text=True,
                       cwd=str(LUA_TEST.parent.parent.parent), timeout=120)
    _out = (r.stdout or "")
    if r.stderr:
        _out = _out + "\n" + r.stderr
    _out = _out.strip()

    # PARSE THE WHOLE OUTPUT; KEEP ONLY THE TAIL FOR CONTEXT.
    # The previous version stored `_out[-2500:]` and then printed
    # `splitlines()[-20:]` of THAT, so run 1789029836 reported "3 FAILURE(S)",
    # showed two of them, and raised an exception about the third -- which it
    # had just truncated away. Selection is by CONTENT now: every line carrying
    # [FAIL] is kept wherever it sits in the output.
    _lines = _out.splitlines()
    _fails = [ln.strip() for ln in _lines if "[FAIL]" in ln]
    _passes = [ln for ln in _lines if "[PASS]" in ln]
    _reported = re.search(r"(\d+)\s+FAILURE", _out)
    GATE7.update(ran=True, ok=(r.returncode == 0), detail=_out[-2500:],
                 failures=_fails, passes=len(_passes),
                 reported_failures=int(_reported.group(1)) if _reported else None)

    # THE COUNT THE MARKDOWN ABOVE ASKS ABOUT, ACTUALLY COUNTED.
    # The old regex looked for the word "assertion". This harness never prints
    # it, so `assertions` was None on every run this cell has ever made -- a
    # dead instrument reporting on the 16-vs-46 question it was built to
    # settle. The assertions ARE the [PASS]/[FAIL] lines.
    _named = re.search(r"(\d+)\s+assertion", _out)
    GATE7["assertions"] = (int(_named.group(1)) if _named
                           else len(_passes) + len(_fails))

    # A FAILURE YOU CANNOT READ IS A FAILURE YOU CANNOT CLASSIFY. If the
    # harness says N and fewer than N [FAIL] lines are visible, something is
    # hidden and the scoped list must not be trusted to cover it.
    GATE7["countable"] = (GATE7["reported_failures"] is None
                          or GATE7["reported_failures"] == len(_fails))
    GATE7["unscoped"] = [ln for ln in _fails if _scope(ln) is None]
    GATE7["scoped"] = (not GATE7["ok"] and GATE7["countable"]
                       and bool(_fails) and not GATE7["unscoped"])

#: Stamped into every document this run writes. A run measured against a filter
#: that fails its own schema test is still a valid run -- the numbers are correct
#: for the filter that is RUNNING -- but only while the documents say so, every
#: time, without anyone having to remember to mention it.
GATE7_PROVENANCE = ""
if GATE7["ran"] and not GATE7["ok"]:
    GATE7_PROVENANCE = (
        f"Measured against a deployed filter that FAILS {len(GATE7['failures'])} of its own "
        f"v3 assertions ({GATE7['source']}). It reports SCHEMA_VERSION=3 and does not "
        "implement all of v3 -- no `transaction` resource kind, and an empty string where an "
        "idle window requires min/max_record_time absent. Both are recorded local-k8s work. "
        "Every number below is correct for the filter that is running, NOT for v3.")
elif not GATE7["ran"]:
    GATE7_PROVENANCE = ("Gate 7 did not run, so the filter is UNPROVEN, not passed: no gate "
                        "result below can be attributed to the query rather than the filter.")

_mark = ("PASS" if GATE7["ok"] else
         ("FAIL (scoped)" if GATE7["scoped"] else ("FAIL" if GATE7["ran"] else "UNPROVEN")))
print(f"  Gate 7  {_mark}")

if GATE7["ran"]:
    print(f"          assertions seen: {GATE7['assertions']}   "
          f"({GATE7['passes']} PASS, {len(GATE7['failures'])} FAIL; the harness reported "
          f"{GATE7['reported_failures'] if GATE7['reported_failures'] is not None else 'no'}"
          f" failure(s))")
    print("          SCHEMA-report.md says 16, GUIDE-schema-v3-testing.md says 46 -- "
          "counted, recorded, not asserted on")
    for _ln in GATE7["failures"]:
        print(f"          {_ln}" + ("   <-- known, scoped" if _scope(_ln) else
                                    "   <-- UNSCOPED"))
    if not GATE7["countable"]:
        print(f"          !! the harness reported {GATE7['reported_failures']} failure(s) and "
              f"{len(GATE7['failures'])} [FAIL] line(s) are visible.")
        print("             One is hidden, so none of them can be classified.")
else:
    print(f"          {GATE7['detail']}")

if not GATE7["ran"]:
    print()
    print("  Gate 7 is UNPROVEN, not passed. The rest still runs, and the findings cell records")
    print("  that the filter's own unit test never executed -- so a gate failure below cannot be")
    print("  attributed to the query rather than to the filter.")
elif GATE7["ok"]:
    pass
elif GATE7["scoped"]:
    print()
    print("  Gate 7 FAILED and every failure is on the recorded list, so the run CONTINUES:")
    for _pattern, _reason in GATE7_SCOPED:
        if any(re.search(_pattern, _ln) for _ln in GATE7["failures"]):
            print(f"    - {_reason}")
    print("  These are defects in the DEPLOYED FILTER, already measured, and they are carried")
    print("  into the findings cell as local-k8s work. Gates 1-6 read requests, writes,")
    print("  last_write_bytes and auth_denied; none of them touch these fields. Aborting here")
    print("  would cost every gate below over a defect that bears on none of them -- which is")
    print("  what run 1789029836 did, leaving cells 5-42 unexecuted.")
else:
    raise RuntimeError(
        "Gate 7 FAILED against the DEPLOYED script text (" + str(GATE7["source"]) + ") with "
        + (f"{len(GATE7['unscoped'])} failure(s) that are NOT on the recorded list: "
           + " | ".join(GATE7["unscoped"])
           if GATE7["unscoped"] else
           f"a failure count that does not reconcile: the harness reported "
           f"{GATE7['reported_failures']}, {len(GATE7['failures'])} are visible")
        + ". Every failure line is printed above. This separates 'the filter is wrong' from "
        "'my query is wrong' in one step, and every gate below would otherwise be measuring a "
        "filter that is known broken in a way nobody has looked at. If the failure names a "
        "missing function or a nil global, check that the ConfigMap key above is the WHOLE "
        "script rather than one fragment of a multi-key chart. If it is a real and understood "
        "pipeline defect that no gate below reads, add it to GATE7_SCOPED with its reason AND "
        "to REPORT-for-local-k8s.md -- never to the first alone.")

  extracted ConfigMap key polaris_access_log.lua (35632 bytes) -> /tmp/polaris.lua
  Gate 7  PASS
          assertions seen: 58   (58 PASS, 0 FAIL; the harness reported no failure(s))
          SCHEMA-report.md says 16, GUIDE-schema-v3-testing.md says 46 -- counted, recorded, not asserted on


## 0b -- The policy that is RUNNING, read from the cluster

`load_policy(FB_VALUES_PATH)` reads the **Deployment's** values file, which is schema v2. This
notebook measures the **DaemonSet**, which is v3. Handing the v2 oracle to a v3 run reproduces
2026-09-04 exactly: eight "unexpected fields" per row, every row correct, nothing naming the
cause.

So the policy is read out of the running ConfigMap, and **both** shippers' versions are printed.
The ConfigMap is not the process either -- a pod that predates it is running the old text -- so
the pod start times are printed beside them.

In [3]:
SHIPPERS = {
    "benchmarks-fluent-bit": {"role": "DaemonSet -> OpenSearch  (THIS notebook)"},
    "fb-polaris-shipper": {"role": "Deployment -> VictoriaLogs  (v1's notebook)"},
}

import re

for rel, info in SHIPPERS.items():
    cm = sh("kubectl", "-n", "datahub-hynix", "get", "cm",
            "-l", f"app.kubernetes.io/instance={rel}", "-o", "yaml") or ""
    body = re.sub(r"--\[\[.*?\]\]", "", cm, flags=re.S)
    body = "\n".join(re.sub(r"--.*$", "", ln) for ln in body.splitlines())
    for const in ("SCHEMA_VERSION", "WINDOW_SECONDS", "REPORT_MAX_ROLE_KEYS"):
        hit = re.search(rf"local\s+{const}\s*=\s*(\d+)", body)
        info[const] = int(hit.group(1)) if hit else None
    info["pods"] = sh("kubectl", "-n", "datahub-hynix", "get", "pods",
                      "-l", f"app.kubernetes.io/instance={rel}",
                      "-o", "jsonpath={range .items[*]}{.metadata.name}@{.status.startTime} {end}")
    info["cm"] = cm
    print(f"  {rel:24} {info['role']}")
    print(f"  {'':24} SCHEMA_VERSION={info['SCHEMA_VERSION']}  "
          f"WINDOW_SECONDS={info['WINDOW_SECONDS']}  ROLE_KEYS={info['REPORT_MAX_ROLE_KEYS']}")
    print(f"  {'':24} {info['pods'] or 'kubectl unavailable -- a mid-run restart will NOT be detectable'}")

TARGET = SHIPPERS["benchmarks-fluent-bit"]
WINDOW_SECONDS = TARGET["WINDOW_SECONDS"]
DEPLOYED_SCHEMA = TARGET["SCHEMA_VERSION"]

if not WINDOW_SECONDS or not DEPLOYED_SCHEMA:
    raise RuntimeError("could not read WINDOW_SECONDS / SCHEMA_VERSION out of the running "
                       "ConfigMap. Everything below indexes windows by that constant; a caller "
                       "that guesses crosses no boundary and gets an EMPTY result rather than "
                       "an error.")
if DEPLOYED_SCHEMA != osr.SCHEMA_VERSION:
    raise RuntimeError(
        f"THE ORACLE IS A SCHEMA VERSION AWAY FROM THE FILTER: the DaemonSet emits v"
        f"{DEPLOYED_SCHEMA}, os_report.SCHEMA_VERSION is {osr.SCHEMA_VERSION}. Every "
        "'unexpected field' below would be that, and not drift in the pipeline.")

OS.schema_version = DEPLOYED_SCHEMA
print(f"\n  measuring schema v{DEPLOYED_SCHEMA}, windows of {WINDOW_SECONDS}s")
if SHIPPERS["fb-polaris-shipper"]["SCHEMA_VERSION"] != DEPLOYED_SCHEMA:
    print("  NOTE the two shippers disagree. That is expected as of 2026-09-10 and is exactly "
          "why\n       the policy is read per-shipper rather than from FB_VALUES_PATH.")

  benchmarks-fluent-bit    DaemonSet -> OpenSearch  (THIS notebook)
                           SCHEMA_VERSION=3  WINDOW_SECONDS=30  ROLE_KEYS=100
                           benchmarks-fluent-bit-d94qc@2026-09-14T05:54:08Z
  fb-polaris-shipper       Deployment -> VictoriaLogs  (v1's notebook)
                           SCHEMA_VERSION=2  WINDOW_SECONDS=30  ROLE_KEYS=None
                           fb-polaris-shipper-fluent-bit-55b7bf586d-5kt5l@2026-09-07T04:21:51Z

  measuring schema v3, windows of 30s
  NOTE the two shippers disagree. That is expected as of 2026-09-10 and is exactly why
       the policy is read per-shipper rather than from FB_VALUES_PATH.


## 0c -- Gate 0: is v3 actually running

**PASS** -- a v3 document comes back and its `window_start` is recent.
**FAILS IF** -- no hits: the ConfigMap changed but the pod never rolled, or the roll never
happened. Reading `schema_version` out of a values file proves nothing, which is why this asks
the index.

Then the two mapping checks that decide whether any *number* below means anything.

In [ ]:
newest = OS.latest_summary()
if not newest:
    raise RuntimeError(
        f"GATE 0 FAILED: no schema_version={OS.schema_version} document in "
        f"{OS.report_index}. The ConfigMap carries v{DEPLOYED_SCHEMA} and the index does not, "
        "so the pod never rolled or the report output is not wired to OpenSearch.")
print(f"  Gate 0  PASS   newest v{OS.schema_version} row: seq={newest.get('report_seq')} "
      f"window={newest.get('window_start')} host={newest.get('hostname')}")

# THE CHECK THAT DECIDES WHETHER EVERY SUM BELOW IS A NUMBER OR A LIE.
# `type_int_key` is set in the filter, but the INDEX decides, and it decides from
# the first document it ever saw. A text-typed `requests` makes gates 1-5 wrong
# while every query still returns 200.
NUMERIC_OK, KEYWORD_OK, MAPPING_NOTES = None, None, []
live = sorted([k for k in IDX if k.startswith("polaris-report")])[-1:]
live = live[0] if live else OS.report_index

# READ THE FIELD NAMES OFF THE LIVE MAPPING BEFORE BUILDING A SINGLE QUERY.
# Run 1789007773 assumed them and lost eight gates to it: `window_start` is an
# RFC3339 string that OpenSearch DATE-DETECTS, so `window_start.keyword` does
# not exist, so every window-scoped report query matched nothing and every gate
# that read one went VOID on an empty result. `min_record_time` is the same
# shape of value and IS text, because the Lua writes "" on idle windows and the
# index typed it from the first one. Two fields, one index, different mappings:
# nothing here can be assumed.
FIELD_NAMES, FIELD_TYPES = OS.learn_field_names(index=live)
print(f"  fields   resolved {len(FIELD_NAMES)} name(s) from {live}")
for _f, _t in sorted(FIELD_TYPES.items()):
    if _t is not None and _t != "text":
        print(f"           {_f:22} mapped {_t:9} -> matched as `{FIELD_NAMES[_f]}`")
try:
    OS.assert_numeric(index=live)
    NUMERIC_OK = True
    print(f"  mapping PASS   requests / errors / response_bytes are numeric in {live}")
except osr.MappingFault as exc:
    NUMERIC_OK = False
    MAPPING_NOTES.append(str(exc))
    print(f"  mapping FAIL   {exc}")

try:
    OS.assert_keyword_ready(index=live)
    KEYWORD_OK = True
except osr.MappingFault as exc:
    KEYWORD_OK = False
    MAPPING_NOTES.append(str(exc))
    print(f"  keyword FAIL   {exc}")

# The guide's own trap, checked rather than remembered.
MIN_RECORD_TYPE = OS.mapping_of("min_record_time", live)
print(f"  min_record_time mapped as {MIN_RECORD_TYPE!r} in {live}"
      + ("   <-- text: no date maths in this index, permanently (review #2)"
         if MIN_RECORD_TYPE not in ("date", None) else ""))
if MIN_RECORD_TYPE not in ("date", None):
    MAPPING_NOTES.append(
        f"min_record_time is {MIN_RECORD_TYPE} in {live}; the invariant is still checkable "
        "client-side (RFC3339 parses), but no range query and no date histogram works on it "
        "in this index.")

if NUMERIC_OK is False:
    raise RuntimeError("the report fields are not numeric in the live index. Every sum in "
                       "gates 1-5 would be meaningless; fix the index template first.")

## 1 -- Identities

Four, and the matrix needs all four. One identity would make the per-principal margin
unfalsifiable: `sum(principal.requests) == access_seen - parse_errors` is satisfied identically
by a global counter when there is only one row to sum.

In [5]:
from polaris_rest import PolarisREST

adm_pc = PolarisREST(ptu.POLARIS_URL, ptu.REALM, token=ROOT_TOK)

RUNNER_NAME, RUNNER_ROLE = f"mx_{RUN}_runner", f"mx_{RUN}_runner_role"
DENIED_NAME, DENIED_ROLE = f"mx_{RUN}_denied", f"mx_{RUN}_denied_role"

TOKENS, IDENTITY_NOTES = {mx.ADMIN: ROOT_TOK}, []

# THE RUNNER IS ELECTED IN THE FIXTURE CELL, NOT HERE, and this is the whole
# lesson of run 1789007773. `provision_run_principal` grants the probe
# catalog's rights only when it is handed the fixture -- its own docstring says
# so: "catalog operations authorize against grant_records for the TARGET
# catalog and service_admin does not cover them". Provisioned here, before the
# fixture exists, the runner holds a role and no privileges, and 19 of the 2xx
# cells came back 403. The matrix then reported a Polaris that refuses every
# catalog operation, which is a statement about this cell.

_cid, _sec = lc.provision_run_principal(adm_pc, DENIED_NAME, DENIED_ROLE, fixture=None,
                                        service_admin=False)
_r = adm_pc.get_token(_cid, _sec, scope=f"PRINCIPAL_ROLE:{DENIED_ROLE}")
if _r.status_code == 200:
    TOKENS[mx.DENIED] = _r.json()["access_token"]
    IDENTITY_NOTES.append(f"denied  {DENIED_NAME} (a role, and no grants at all)")
else:
    TOKENS[mx.DENIED] = mx.GARBAGE_TOKEN
    IDENTITY_NOTES.append(f"denied  NO TOKEN ({_r.status_code}) -- every 403 cell will report "
                          "401 instead and be MISSED, correctly")

for n in IDENTITY_NOTES:
    print(f"  {n}")
print("  nobody  a syntactically valid bearer no realm accepts")

  denied  mx_1789370776_denied (a role, and no grants at all)
  nobody  a syntactically valid bearer no realm accepts


## 2 -- The fixture, and the binding every cell is shaped against

One catalog, namespace, table and view. The `binding` is what turns a spec template into a URL;
`bind_path` raises rather than leaving a `{param}` behind, because a literal brace in a request
path is a 404 that looks like a finding.

In [6]:
import api_surface as surf
from iceberg_rest import IcebergREST, build_create_table_payload, build_schema

adm_ic = IcebergREST(ptu.POLARIS_URL, ptu.REALM, token=ROOT_TOK)
run_adm_ic = adm_ic  # the doomed family is created as root; the grid deletes it as itself
SCHEMA = build_schema([(1, "id", "long", True), (2, "val", "string", False)])

FX = surf.ProbeFixture(prefix=f"apimatrix{RUN}")
surf.setup_fixture(FX, adm_pc, adm_ic, ptu.BUCKET, ptu.MINIO_ENDPOINT, SCHEMA,
                   build_create_table_payload)
print(f"  fixture  {FX.cat} / {FX.ns} / {FX.tbl} / {FX.view}")

# NOW the runner can be elected: elect_drive_identity provisions it WITH the
# fixture (which is what grants the catalog rights) and then VERIFIES it with
# one management read and one catalog read, demoting to root loudly if it
# cannot do both. Run 1789007773 skipped both halves and spent 286 calls
# measuring a principal that could not read a table.
IDENTITY, run_pc, run_ic_elected, DRIVE_PRINCIPAL, IDENTITY_NOTES = lc.elect_drive_identity(
    adm_pc, adm_ic, ptu.POLARIS_URL, ptu.REALM, RUNNER_NAME, RUNNER_ROLE, fixture=FX,
    root_token=ROOT_TOK)
TOKENS[mx.RUNNER] = run_pc.token or ROOT_TOK
for _n in IDENTITY_NOTES:
    print(f"  identity {_n}")
print(f"  runner   {DRIVE_PRINCIPAL} (elected: {IDENTITY})")
if DRIVE_PRINCIPAL == "root" or TOKENS[mx.RUNNER] == ROOT_TOK:
    print("  !! THE RUN IS DRIVING AS ROOT. That is a valid fallback and it VOIDS the")
    print("     per-principal margin: sum(principal.requests) == access_seen - parse_errors")
    print("     is then satisfied by a global counter with one row to sum, and proves nothing.")

BINDING = {
    "catalog": FX.cat, "namespace": FX.ns, "table": FX.tbl, "view": FX.view,
    "principalName": RUNNER_NAME, "principalRoleName": RUNNER_ROLE,
    "catalogRoleName": f"mx_{RUN}_crole", "plan-id": "no-such-plan-id",
    # names the grid CREATES rather than reads
    "new_catalog": f"mx{RUN}cat2", "new_namespace": f"mx_{RUN}_ns2",
    "new_table": f"mx_{RUN}_tbl2", "new_view": f"mx_{RUN}_vw2",
    "new_principal": f"mx_{RUN}_p2", "new_principal_role": f"mx_{RUN}_prole2",
    "new_catalog_role": f"mx_{RUN}_crole2",
    "renamed_table": f"{FX.tbl}_renamed", "renamed_view": f"{FX.view}_renamed",
    # THE DOOMED FAMILY. Every destructive operation in the grid is pointed
    # here by api_status_matrix.REBIND, and nothing else in the run reads them.
    "doomed_catalog": f"mx{RUN}doomed", "doomed_namespace": f"mx_{RUN}_ns_doomed",
    "doomed_table": f"mx_{RUN}_tbl_doomed", "doomed_view": f"mx_{RUN}_vw_doomed",
    "doomed_principal": f"mx_{RUN}_p_doomed",
    "doomed_principal_role": f"mx_{RUN}_prole_doomed",
    "doomed_catalog_role": f"mx_{RUN}_crole_doomed",
    "entity_version": 1, "run": RUN,
    "base_location": f"s3a://{ptu.BUCKET}/mx{RUN}cat2/",
    "allowed_location": f"s3a://{ptu.BUCKET}/",
    "s3_endpoint": ptu.MINIO_ENDPOINT, "s3_endpoint_internal": ptu.MINIO_ENDPOINT_INTERNAL,
    "metadata_location": f"s3a://{ptu.BUCKET}/mx{RUN}cat2/never-registered/metadata.json",
    "client_id": ptu.ROOT_CLIENT, "client_secret": ptu.ROOT_SECRET,
}

# The catalog role the role-graph cells READ has to exist before they read it.
_r = adm_pc.create_catalog_role(FX.cat, BINDING["catalogRoleName"])
print(f"  catalog role {BINDING['catalogRoleName']}: {_r.status_code}")

# THE DOOMED FAMILY. The happy sweep drives deleteCatalog, dropTable, dropView,
# dropNamespace, deletePrincipal and the two revokes -- bound naively they take
# the FIXTURE, and the run destroys itself at about call 30 of 63 with every
# later cell 404ing. api_status_matrix.REBIND points them here instead, so the
# deletes are real deletes of real entities and nothing the run still needs.
SETUP = {}
SETUP["doomed catalog"] = adm_pc.create_catalog(
    BINDING["doomed_catalog"], ptu.BUCKET, ptu.MINIO_ENDPOINT,
    minio_endpoint_internal=ptu.MINIO_ENDPOINT_INTERNAL,
).status_code
SETUP["doomed namespace"] = run_adm_ic.create_namespace(
    FX.cat, BINDING["doomed_namespace"]).status_code
SETUP["doomed table"] = run_adm_ic.create_table(
    FX.cat, FX.ns, build_create_table_payload(BINDING["doomed_table"], SCHEMA)).status_code
SETUP["doomed view"] = run_adm_ic.create_view(FX.cat, FX.ns, {
    "name": BINDING["doomed_view"], "schema": SCHEMA,
    "view-version": {"version-id": 1, "timestamp-ms": 0, "schema-id": 0,
                     "summary": {"engine-name": "api-status-matrix"},
                     "default-namespace": [FX.ns],
                     "representations": [{"type": "sql", "sql": "SELECT 1",
                                          "dialect": "spark"}]},
    "properties": {"mx.run": RUN}}).status_code
SETUP["doomed principal"] = adm_pc.create_principal(BINDING["doomed_principal"]).status_code
SETUP["doomed principal role"] = adm_pc.create_principal_role(
    BINDING["doomed_principal_role"]).status_code
SETUP["doomed catalog role"] = adm_pc.create_catalog_role(
    FX.cat, BINDING["doomed_catalog_role"]).status_code
# the two revokes need something assigned to revoke
SETUP["assign prole->principal"] = adm_pc.assign_principal_role_to_principal(
    BINDING["doomed_principal"], BINDING["doomed_principal_role"]).status_code
# ARGUMENT ORDER IS (catalog, principal_role, catalog_role) -- and this PUT can
# 404 immediately after create_catalog_role on PG-HA read-after-write lag, which
# is a known signature here, not a finding. Retry rather than record a false one.
SETUP["assign crole->prole"] = None
for _try in range(4):
    _ar = adm_pc.assign_catalog_role_to_principal_role(
        FX.cat, BINDING["doomed_principal_role"], BINDING["doomed_catalog_role"])
    SETUP["assign crole->prole"] = _ar.status_code
    if _ar.status_code != 404:
        break
    time.sleep(0.75)

for k, v in SETUP.items():
    print(f"  {k:26} {v}")
_bad = {k: v for k, v in SETUP.items() if not (200 <= int(v) < 300)}
if _bad:
    print(f"  !! setup did not fully succeed: {_bad}")
    print("     the delete cells for those entities will report 404 and be MISSED --")
    print("     correctly, and as a harness gap rather than a pipeline finding.")

# TEARDOWN THAT SURVIVES AN ABORT.
#
# Cleanup is section 11, eighteen cells below, and until now it was reachable
# only by running every cell between here and there. Any raise in between -- a
# gate, a transport error, an interrupted kernel -- left the catalog, the
# namespace, the table, the view, both principals and the whole doomed family
# on the cluster. That is where `apimatrix1789031469_cat` and `probe_ns` came
# from, and a leaked catalog is not tidiness: the next run's fixture collides
# with it and the collision is reported as a finding about Polaris.
#
# `teardown_all` is idempotent, never raises, and reads its clients out of
# globals(), so it is correct whether it runs from section 11, from atexit on
# kernel shutdown, or by hand in a fresh cell after a failure. Deletion order
# is children first: views and tables, then namespaces, then catalog roles,
# then catalogs. A 404 here is the expected answer for anything the grid
# already deleted as part of the test, and it is printed rather than hidden.
import atexit

_TORN = {"done": False}


def teardown_all(quiet=False):
    """Delete everything this run created. Idempotent, best-effort, never raises."""
    if _TORN["done"]:
        if not quiet:
            print("  teardown already ran -- nothing to do")
        return {}
    _TORN["done"] = True
    g = globals()
    ic = g.get("run_ic") or g.get("adm_ic")
    pc = g.get("adm_pc")
    out = {}

    def _try(label, fn):
        try:
            out[label] = fn().status_code
        except Exception as exc:
            out[label] = f"{type(exc).__name__}: {exc}"
        if not quiet:
            print(f"  {label:34} {out[label]}")

    if ic is not None:
        # THE RENAME DESTINATIONS, and leaving them out is what kept
        # `apimatrix1789365275_cat` on the cluster: renameTable/renameView
        # move `new_table`/`new_view` onto these names, nothing drops them,
        # so probe_ns answered 409 and the catalog delete 400. A teardown
        # that does not know every name the RUN can create is not a teardown.
        for _v in (FX.view, BINDING["doomed_view"], BINDING["new_view"],
                   BINDING["renamed_view"]):
            _try(f"drop view {_v}", lambda v=_v: ic.drop_view(FX.cat, FX.ns, v))
        for _t in (FX.tbl, BINDING["doomed_table"], BINDING["new_table"],
                   BINDING["renamed_table"]):
            _try(f"drop table {_t}", lambda t=_t: ic.drop_table(FX.cat, FX.ns, t))
        for _n in (BINDING["doomed_namespace"], BINDING["new_namespace"], FX.ns):
            _try(f"drop namespace {_n}", lambda n=_n: ic.drop_namespace(FX.cat, n))

    # THE SWEEP. Deleting a fixed list of names cannot be complete, because the
    # RUN creates entities it did not intend to: run 1789366561 left
    # `apimatrix..._cat` undeletable (400) after every named entity was gone,
    # and the likeliest reason is in its own findings -- `createNamespace`
    # answered 500 and COMMITTED anyway. A teardown that only knows the names it
    # planned will always trail the run by whatever the run did by accident.
    if ic is not None:
        try:
            _ns_resp = ic.list_namespaces(FX.cat)
            _nss = [n for n in (_ns_resp.json().get("namespaces") or [])] if _ns_resp.ok else []
        except Exception as exc:
            _nss = []
            if not quiet:
                print(f"  {'sweep: list namespaces':34} {type(exc).__name__}: {exc}")
        for _levels in _nss:
            _ns = ".".join(_levels) if isinstance(_levels, list) else str(_levels)
            for _kind, _lister, _dropper in (
                ("view", ic.list_views, ic.drop_view),
                ("table", ic.list_tables, ic.drop_table),
            ):
                try:
                    _r = _lister(FX.cat, _ns)
                    _ids = (_r.json().get("identifiers") or []) if _r.ok else []
                except Exception:
                    _ids = []
                for _id in _ids:
                    _nm = _id.get("name") if isinstance(_id, dict) else str(_id)
                    _try(f"sweep {_kind} {_ns}/{_nm}", lambda n=_ns, m=_nm, d=_dropper: d(FX.cat, n, m))
            _try(f"sweep namespace {_ns}", lambda n=_ns: ic.drop_namespace(FX.cat, n))

    if pc is not None:
        for _cr in (BINDING["catalogRoleName"], BINDING["doomed_catalog_role"],
                    BINDING["new_catalog_role"]):
            _try(f"delete catalog role {_cr}", lambda c=_cr: pc.delete_catalog_role(FX.cat, c))
        for _cat in (FX.cat, BINDING["doomed_catalog"], BINDING["new_catalog"]):
            _try(f"delete catalog {_cat}", lambda c=_cat: pc.delete_catalog(c))
        for _p in (BINDING["doomed_principal"], BINDING["new_principal"]):
            _try(f"delete principal {_p}", lambda p=_p: pc.delete_principal(p))
        for _pr in (BINDING["doomed_principal_role"], BINDING["new_principal_role"]):
            _try(f"delete principal role {_pr}",
                 lambda p=_pr: pc.delete_principal_role(p))
        for _name, _role in ((RUNNER_NAME, RUNNER_ROLE), (DENIED_NAME, DENIED_ROLE)):
            try:
                lc.deprovision_run_principal(pc, _name, _role)
                out[f"deprovision {_name}"] = "ok"
            except Exception as exc:
                out[f"deprovision {_name}"] = f"{type(exc).__name__}: {exc}"
            if not quiet:
                print(f"  {'deprovision ' + _name:34} {out['deprovision ' + _name]}")
    return out


def _teardown_at_exit():
    if not _TORN["done"]:
        try:
            teardown_all(quiet=True)
        except Exception:
            pass


atexit.register(_teardown_at_exit)
print("  teardown registered: teardown_all() cleans up from anywhere, and atexit runs it on")
print("                       kernel shutdown if the run aborts before section 11")

  fixture  apimatrix1789370776_cat / probe_ns / probe_tbl / probe_view
  identity driving as mx_1789370776_runner (scope=PRINCIPAL_ROLE:ALL)
  runner   mx_1789370776_runner (elected: mx_1789370776_runner)
  catalog role mx_1789370776_crole: 201
  doomed catalog             201
  doomed namespace           200
  doomed table               200
  doomed view                200
  doomed principal           201
  doomed principal role      201
  doomed catalog role        201
  assign prole->principal    201
  assign crole->prole        201
  teardown registered: teardown_all() cleans up from anywhere, and atexit runs it on
                       kernel shutdown if the run aborts before section 11


## 3 -- The grid, printed BEFORE anything is driven

The denominator is a fact about the vendored specs and does not need this run to establish it.
Printing it first means a reader can see what was planned even if the drive falls over halfway.

The **not-reachable ledger** prints beside it. `settled` entries have a structural reason;
`probe` entries are assumptions that get one call each, recorded either way.

In [7]:
import collections

OPS = mx.load_spec(pathlib.Path.cwd() / "spec")
GRID = mx.cells(OPS)
BY_TARGET = collections.Counter(c.target for c in GRID)
BY_PHASE = collections.Counter(c.phase for c in GRID)

print(f"  {len(OPS)} operations from log-coverage/spec/  "
      f"({sum(1 for o in OPS if o.api == 'management')} management, "
      f"{sum(1 for o in OPS if o.api == 'catalog')} catalog)")
print(f"  {len(GRID)} cells   " + "  ".join(f"{k}={v}" for k, v in sorted(BY_TARGET.items(), key=str)))
print("  phases   " + "  ".join(f"{k}={v}" for k, v in sorted(BY_PHASE.items())))

print("\n  NOT REACHABLE on this build:")
for row in mx.ledger():
    mark = "settled" if row["state"] == "settled" else "PROBE  "
    print(f"    {str(row['status']):>4}  {mark}  {row['why']}")
print(f"  EXTRAS the spec never declares but this build returns: {mx.UNDECLARED_EXTRAS}")

# Every cell must build a request before ANY of them is issued. A missing payload
# or an unbindable path is a harness bug, and finding it at call 200 of 286 wastes
# the run's windows.
_dry = [mx.request_for(c, BINDING, TOKENS, "dry-run", ptu.REALM) for c in GRID]
print(f"\n  dry run  {len(_dry)} requests built, none raised")

ROWS, PHASE_WINDOWS = [], {}


def run_phase(name, cells_, note=""):
    """Drive one phase, alone in its window, and record the window it fell in."""
    wait = osr.seconds_to_boundary(WINDOW_SECONDS, lag=0.5)
    print(f"  phase {name}: {len(cells_)} cells; waiting {wait:.0f}s for a fresh window {note}")
    time.sleep(wait)
    t0 = time.time()
    rows = mx.drive(cells_, BINDING, TOKENS, ptu.POLARIS_URL, ptu.REALM, RUN,
                    seq_from=2000 + len(ROWS))
    t1 = time.time()
    w0, _ = osr.window_bounds(t0, WINDOW_SECONDS)
    w1, _ = osr.window_bounds(t1, WINDOW_SECONDS)
    PHASE_WINDOWS[name] = {"start": w0, "end": w1, "straddled": w0 != w1,
                           "seconds": round(t1 - t0, 1), "cells": len(cells_)}
    ROWS.extend(rows)
    got = collections.Counter(r["verdict"] for r in rows)
    print(f"           {t1 - t0:.1f}s in window {w0}"
          + ("  !! STRADDLED -- this phase's window checks are VOID, not failed"
             if w0 != w1 else "")
          + f"   {dict(got)}")
    return rows

  63 operations from log-coverage/spec/  (33 management, 30 catalog)
  286 cells   2=63  400=28  401=63  403=62  404=55  409=15
  phases   B=63  C=180  D=43

  NOT REACHABLE on this build:
     304  PROBE    needs If-None-Match support on loadTable
     406  PROBE    needs content negotiation to refuse a JSON-only endpoint
     419  PROBE    Iceberg's credential-refresh code; this build is expected to answer 401 instead
     429  settled  rateLimiter.type is no-op on this build -- the limiter cannot produce a 429
     502  settled  nothing proxies Polaris here; a port-forward is not a gateway
     503  settled  the only route is scaling Polaris, and HPA movement invalidates the run (local-k8s #8)
     504  settled  same -- no gateway to time out
     5XX  settled  a spec placeholder, not a status
  EXTRAS the spec never declares but this build returns: (405, 500)

  dry run  286 requests built, none raised


## 4 -- Phase B: the 2xx sweep

Every operation at its own success code, adjudicated against what the spec **declares** --
`createCatalog` answers 201 and `dropNamespace` 204, so a matrix expecting 200 would call both
a miss.

In [8]:
PHASE_B = run_phase("B", [c for c in GRID if c.phase == "B" and c.target == 2])
_missed = [r for r in PHASE_B if r["verdict"] == mx.MISSED]
print(f"\n  {len(PHASE_B) - len(_missed)}/{len(PHASE_B)} covered")
for r in _missed[:20]:
    print(f"    MISS  {r['op_id']:34} got {r['status']}  {r['why']}")

  phase B: 63 cells; waiting 11s for a fresh window 
           1.0s in window 2026-09-14T07:26:30Z   {'covered': 50, 'missed': 13}

  50/63 covered
    MISS  registerTable                      got 400  expected one of (200,)
    MISS  planTableScan                      got 404  expected one of (200,)
    MISS  cancelPlanning                     got 404  expected one of (204,)
    MISS  fetchPlanningResult                got 404  expected one of (200,)
    MISS  fetchScanTasks                     got 404  expected one of (200,)
    MISS  loadView                           got 404  expected one of (200,)
    MISS  viewExists                         got 404  expected one of (204,)
    MISS  replaceView                        got 404  expected one of (200,)
    MISS  renameTable                        got 200  expected one of (204,)
    MISS  revokeGrantFromCatalogRole         got 400  expected one of (201,)
    MISS  revokeCatalogRoleFromPrincipalRole got 404  expected one of (204,)
    

## 5 -- Phase C: 401 / 403 / 404

The error sweep. Rule 3 keeps **every one of these** as an individual record, so this is also
the phase that proves the per-call half of the pipeline: every cell here must be findable in
`polaris-logs-*` by its request id.

Gate 5 reads this window: these errors land in `__errors__` (attribution lost, record kept),
and `__other__` must stay absent or zero -- no window here carries 500 distinct resources.

In [9]:
PHASE_C = run_phase("C", [c for c in GRID if c.phase == "C"])
_by = collections.Counter((r["target"], r["status"]) for r in PHASE_C)
print("\n  target -> actual:")
for (t, a), n in sorted(_by.items(), key=lambda kv: (str(kv[0][0]), str(kv[0][1]))):
    mark = "" if t == a else "   <-- MISS"
    print(f"    {t:>4} -> {str(a):>4}  x{n}{mark}")

  phase C: 180 cells; waiting 29s for a fresh window 
           1.0s in window 2026-09-14T07:27:00Z   {'covered': 155, 'missed': 25}

  target -> actual:
     401 ->  401  x59
     401 ->  404  x4   <-- MISS
     403 ->  200  x1   <-- MISS
     403 ->  204  x1   <-- MISS
     403 ->  403  x42
     403 ->  404  x18   <-- MISS
     404 ->  204  x1   <-- MISS
     404 ->  404  x54


## 6 -- Phase D: 409 / 400

409 comes from **order** on the creates -- phase B already made these -- and from a stale
`currentEntityVersion` on the management PUTs. 400 comes from a body missing its required
fields, never from `{}`: several Polaris endpoints accept an empty object and answer 2xx.

In [10]:
PHASE_D = run_phase("D", [c for c in GRID if c.phase == "D"])
for target in (409, 400):
    rows = [r for r in PHASE_D if r["target"] == target]
    hit = sum(1 for r in rows if r["verdict"] == mx.COVERED)
    print(f"  {target}: {hit}/{len(rows)} covered")
    for r in rows:
        if r["verdict"] != mx.COVERED:
            print(f"    MISS  {r['op_id']:34} got {r['status']}")

  phase D: 43 cells; waiting 29s for a fresh window 
           0.5s in window 2026-09-14T07:27:30Z   {'covered': 23, 'missed': 20}
  409: 7/15 covered
    MISS  dropNamespace                      got 404
    MISS  registerTable                      got 400
    MISS  createTable                        got 403
    MISS  updateTable                        got 200
    MISS  createView                         got 200
    MISS  replaceView                        got 404
    MISS  renameTable                        got 404
    MISS  commitTransaction                  got 204
  400: 16/28 covered
    MISS  getToken                           got 500
    MISS  createNamespace                    got 500
    MISS  updateTable                        got 200
    MISS  planTableScan                      got 404
    MISS  fetchScanTasks                     got 404
    MISS  replaceView                        got 404
    MISS  renameTable                        got 500
    MISS  renameView            

## 7 -- Phase E: ONE commit, alone in its window

**Gate 2 is the feature. Everything else tests plumbing.**

`last_write_bytes` on the table row must equal the `response_size` of the **actual commit** that
produced it. "The last 2xx POST in that window" is only unambiguous when the window holds one,
which is why this phase is one call.

**FAILS IF** it matches the *first* commit (last-wins broken), or a 500's size (the 2xx filter
broken), or 0 (the `> 0` guard broken), or the field is absent when a commit did happen.

In [11]:
from iceberg_rest import IcebergREST as _IR

run_ic = _IR(ptu.POLARIS_URL, ptu.REALM, token=TOKENS[mx.RUNNER])

# PHASE F'S FIXTURE, CREATED HERE ON PURPOSE -- before the boundary wait, so
# its CREATE falls in a window neither phase owns. Created at the top of
# section 8 instead it shares this window with the commit, and Gate 2's whole
# premise is that this window holds exactly one write.
_drop_tbl = f"mx_{RUN}_droponly"
_r = run_ic.create_table(FX.cat, FX.ns, build_create_table_payload(_drop_tbl, SCHEMA))
print(f"  created {_drop_tbl}: {_r.status_code}  (in the PREVIOUS window, deliberately)")

_wait = osr.seconds_to_boundary(WINDOW_SECONDS, lag=0.5)
print(f"  waiting {_wait:.0f}s so the commit is alone in its window")
time.sleep(_wait)
_t0 = time.time()
_rid = mx.request_id(RUN, 3001, "gate2-the-commit")
lc.tag_clients([run_ic], _rid)
_resp = run_ic.commit_table(FX.cat, FX.ns, FX.tbl,
                            [{"action": "set-properties", "updates": {"mx.gate2": RUN}}])
lc.tag_clients([run_ic], None)
GATE2 = {
    "request_id": _rid, "status": _resp.status_code,
    "response_size": len(_resp.content or b""),
    "window": osr.window_bounds(_t0, WINDOW_SECONDS)[0],
    "window_end": osr.window_bounds(_t0, WINDOW_SECONDS)[1],
    "table": FX.tbl,
}
PHASE_WINDOWS["E"] = {"start": GATE2["window"], "end": GATE2["window"],
                      "straddled": False, "seconds": round(time.time() - _t0, 1), "cells": 1}
print(f"  commit {GATE2['status']}, {GATE2['response_size']} bytes, window {GATE2['window']}")
print("  NOTE the byte count above is what the CLIENT received. The gate compares the report's")
print("       claim against the ACCESS LOG's response_size, not against this -- they should")
print("       agree, and if they do not the access log is the authority.")

  created mx_1789370776_droponly: 200  (in the PREVIOUS window, deliberately)
  waiting 29s so the commit is alone in its window
  commit 200, 1941 bytes, window 2026-09-14T07:28:00Z
  NOTE the byte count above is what the CLIENT received. The gate compares the report's
       claim against the ACCESS LOG's response_size, not against this -- they should
       agree, and if they do not the access log is the authority.


## 8 -- Phase F: ONE delete, alone in its window

Gate 2's negative half, which is the one people skip. A window whose only write was a `DELETE`
(204, empty body) must carry **no** `last_write_bytes` at all -- **absent, not 0**. A query
looking for `: 0` would pass against a schema doing the thing the schema forbids, so the check
is `must_not exists`.

In [12]:
# The table was created back in section 7, BEFORE that section's boundary wait, so
# it lands at least one full window before the commit. It used to be created here
# -- which put its CREATE in the commit's own window and broke the one premise
# Gate 2 has: run 1789365275 found `droponly writes=1` sitting in phase E's window
# and the commit attributed nowhere, so the gate went VOID.
_wait = osr.seconds_to_boundary(WINDOW_SECONDS, lag=0.5)
print(f"  waiting {_wait:.0f}s so the DELETE is alone")
time.sleep(_wait)
_t0 = time.time()
_rid = mx.request_id(RUN, 3002, "gate2-the-delete")
lc.tag_clients([run_ic], _rid)
_resp = run_ic.drop_table(FX.cat, FX.ns, _drop_tbl)
lc.tag_clients([run_ic], None)
GATE2_NEG = {
    "request_id": _rid, "status": _resp.status_code,
    "response_size": len(_resp.content or b""), "table": _drop_tbl,
    "window": osr.window_bounds(_t0, WINDOW_SECONDS)[0],
}
PHASE_WINDOWS["F"] = {"start": GATE2_NEG["window"], "end": GATE2_NEG["window"],
                      "straddled": False, "seconds": round(time.time() - _t0, 1), "cells": 1}
print(f"  drop {GATE2_NEG['status']}, {GATE2_NEG['response_size']} bytes, "
      f"window {GATE2_NEG['window']}")
if GATE2_NEG["response_size"] != 0:
    print("  !! the DELETE returned a body. The 'size > 0' rule is what separates commit from")
    print("     drop, so a non-empty 204 makes this negative case VOID rather than failed.")

  waiting 30s so the DELETE is alone
  drop 204, 0 bytes, window 2026-09-14T07:28:30Z


## 9 -- Phase G: grants, and a denied read on the same role

Gate 4. **`writes` on a `catalog-role` row is the number of privileges granted in that window** --
grants fold into the role they are granted on, because `classify` returns the matched span.

Then the denial case: a **403** on a catalog role with no successful request in the same window
must **still** produce that role's row, with `auth_denied: 1`, and increment `role_keys_forced`
on the summary. Role kinds are the one exception to the `create=false` guard; if the 403 lands
in `__errors__` instead, the `ROLE_KINDS` exemption is not working.

In [13]:
GRANTS = ["TABLE_READ_DATA", "TABLE_WRITE_DATA", "NAMESPACE_LIST"]
_crole = BINDING["catalogRoleName"]

_wait = osr.seconds_to_boundary(WINDOW_SECONDS, lag=0.5)
print(f"  waiting {_wait:.0f}s")
time.sleep(_wait)
_t0 = time.time()

granted = []
for i, priv in enumerate(GRANTS):
    _rid = mx.request_id(RUN, 3100 + i, f"gate4-grant-{priv}")
    lc.tag_clients([adm_pc], _rid)
    r = adm_pc.grant_privilege(FX.cat, _crole, priv)
    lc.tag_clients([adm_pc], None)
    granted.append({"privilege": priv, "status": r.status_code, "request_id": _rid})
    print(f"    grant {priv:22} {r.status_code}")

# The denied principal reads the role it has no grants on. No successful request
# to this role from this identity in this window -- which is the whole point.
denied_pc = PolarisREST(ptu.POLARIS_URL, ptu.REALM, token=TOKENS[mx.DENIED])
_rid = mx.request_id(RUN, 3199, "gate4-denied-role-read")
lc.tag_clients([denied_pc], _rid)
_dr = denied_pc.get_catalog_role(FX.cat, _crole)
lc.tag_clients([denied_pc], None)
print(f"    denied read of the role: {_dr.status_code}")

GATE4 = {
    "role": _crole, "granted": granted,
    "expected_writes": sum(1 for g in granted if 200 <= (g["status"] or 0) < 300),
    "denied_status": _dr.status_code, "denied_request_id": _rid,
    "window": osr.window_bounds(_t0, WINDOW_SECONDS)[0],
}
PHASE_WINDOWS["G"] = {"start": GATE4["window"], "end": osr.window_bounds(time.time(), WINDOW_SECONDS)[0],
                      "straddled": GATE4["window"] != osr.window_bounds(time.time(), WINDOW_SECONDS)[0],
                      "seconds": round(time.time() - _t0, 1), "cells": len(granted) + 1}
print(f"  expected writes on {_crole}: {GATE4['expected_writes']}  window {GATE4['window']}")

  waiting 30s
    grant TABLE_READ_DATA        201
    grant TABLE_WRITE_DATA       201
    grant NAMESPACE_LIST         201
    denied read of the role: 403
  expected writes on mx_1789370776_crole: 3  window 2026-09-14T07:29:00Z


## 10 -- Phase H: the 500 ladder

`lc.provokers_500`, in its own pure window so `errors_4xx == 0` is a **prediction** rather than
an observation. Every rung answered a **4xx** on 2026-09-07 -- a broken storage endpoint is 422
here, a missing bucket 400, a stale `entityVersion` 409 -- so **NOT PROVOKED is the expected
result and it is the contract working**, not a gap to paper over.

In [14]:
PROVOKERS = lc.provokers_500(
    adm_pc, run_ic, RUN, bucket=ptu.BUCKET, endpoint=ptu.MINIO_ENDPOINT,
    endpoint_internal=ptu.MINIO_ENDPOINT_INTERNAL,
    table_payload=lambda name: build_create_table_payload(name, SCHEMA), repeat=3)
print(f"  ladder: {[p.name for p in PROVOKERS]}")

_wait = osr.seconds_to_boundary(WINDOW_SECONDS, lag=0.5)
print(f"  waiting {_wait:.0f}s so the burst is pure")
time.sleep(_wait)
_t0 = time.time()
_r500 = lc.drive_500([adm_pc, run_ic], RUN, 3200, PROVOKERS,
                     on_rung=lambda e: print(f"    {e['rung']:22} calls={e['calls']} "
                                             f"statuses={e['statuses']} provoked={e['provoked']}"))
_t1 = time.time()

FIVE_HUNDRED = {
    "winner": _r500["winner"], "rows": _r500["rows"], "ladder": _r500["ladder"],
    "driven_500": len(lc.classify_500s(_r500["rows"])["deliberate"]),
    "window": osr.window_bounds(_t0, WINDOW_SECONDS)[0],
    "straddled": osr.window_bounds(_t0, WINDOW_SECONDS)[0]
                 != osr.window_bounds(_t1, WINDOW_SECONDS)[0],
}
PHASE_WINDOWS["H"] = {"start": FIVE_HUNDRED["window"], "end": osr.window_bounds(_t1, WINDOW_SECONDS)[0],
                      "straddled": FIVE_HUNDRED["straddled"],
                      "seconds": round(_t1 - _t0, 1), "cells": len(_r500["rows"])}

if FIVE_HUNDRED["winner"]:
    print(f"  PROVOKED by `{FIVE_HUNDRED['winner']}`: {FIVE_HUNDRED['driven_500']} deliberate 500(s)")
else:
    print("  NOT PROVOKED -- no rung returned >= 500. Expected on this build: "
          "IcebergExceptionMapper\n     catches storage failures and maps them to 4xx. "
          "errors_5xx is NOT exercised this run,\n     and the accidental PG-HA 500s are not a "
          "substitute -- they are writes that COMMITTED.")
if FIVE_HUNDRED["straddled"]:
    print("  !! the burst straddled a boundary: the pure-window checks are VOID. Re-drive rather")
    print("     than reasoning about the split.")

  ladder: ['black_hole_endpoint', 'unresolvable_host', 'nonexistent_bucket', 'stale_entity_version']
  waiting 30s so the burst is pure
    black_hole_endpoint    calls=3 statuses=[422, 422, 422] provoked=0
    unresolvable_host      calls=3 statuses=[422, 422, 422] provoked=0
    nonexistent_bucket     calls=3 statuses=[400, 400, 400] provoked=0
    stale_entity_version   calls=3 statuses=[409, 409, 409] provoked=0
  NOT PROVOKED -- no rung returned >= 500. Expected on this build: IcebergExceptionMapper
     catches storage failures and maps them to 4xx. errors_5xx is NOT exercised this run,
     and the accidental PG-HA 500s are not a substitute -- they are writes that COMMITTED.


## 11 -- Phase I: cleanup

The cleanup DELETEs are **part of the test**, not tidying: `dropTable`, `dropView`,
`dropNamespace`, `deleteCatalog`, `deletePrincipal` are five of the 63 operations and rule 4
keeps every one of them.

In [15]:
# The cleanup DELETEs are **part of the test**, not tidying: `dropTable`,
# `dropView`, `dropNamespace`, `deleteCatalog`, `deletePrincipal` are five of
# the 63 operations and rule 4 keeps every one of them. So they are run here,
# in the open, rather than left to the atexit hook.
#
# `teardown_all` was defined and registered in section 2, the moment the
# fixture existed, precisely so that an abort anywhere between there and here
# still cleans up. This cell is now the NORMAL path through it, not the only
# one. It is idempotent: if the hook already fired, it says so and does nothing.
CLEANUP = teardown_all()

  drop view probe_view               404
  drop view mx_1789370776_vw_doomed  404
  drop view mx_1789370776_vw2        204
  drop view probe_view_renamed       204
  drop table probe_tbl               204
  drop table mx_1789370776_tbl_doomed 404
  drop table mx_1789370776_tbl2      404
  drop table probe_tbl_renamed       204
  drop namespace mx_1789370776_ns_doomed 404
  drop namespace mx_1789370776_ns2   204
  drop namespace probe_ns            204
  sweep namespace probe_ns2          204
  delete catalog role mx_1789370776_crole 204
  delete catalog role mx_1789370776_crole_doomed 404
  delete catalog role mx_1789370776_crole2 204
  delete catalog apimatrix1789370776_cat 400
  delete catalog mx1789370776doomed  404
  delete catalog mx1789370776cat2    204
  delete principal mx_1789370776_p_doomed 404
  delete principal mx_1789370776_p2  204
  delete principal role mx_1789370776_prole_doomed 404
  delete principal role mx_1789370776_prole2 204
  deprovision mx_1789370776_runner   ok

## 12 -- Collection

Every call carried a `Polaris-Request-Id`, so the per-call half is a join rather than a guess.
`settle` waits for the count to stop moving: ingest is asynchronous and a count taken too early
is a statement about the clock.

**Scoped to the run's own windows.** An unscoped count over a continuously written index differs
from itself by elapsed time -- one row per window, two a minute -- which on 2026-09-10 nearly
became a finding about 55 rows that were never missing.

In [16]:
import datetime as _dt

def _rfc3339(v):
    """Parse, tolerating the 9-digit fractional seconds the filter emits.

    `datetime.fromisoformat` is fussy about more than 6 fractional digits on some
    interpreters, and the old code answered a ValueError with `continue` -- so an
    unparseable timestamp silently shrank the denominator instead of failing.
    """
    if not v:
        return None
    t = str(v).strip().replace("Z", "+00:00")
    if "." in t:
        head, _, rest = t.partition(".")
        i = 0
        while i < len(rest) and rest[i].isdigit():
            i += 1
        digits, tz = rest[:i], (rest[i:] or "+00:00")
        t = f"{head}.{digits[:6].ljust(6, '0')}{tz}"
    try:
        return _dt.datetime.fromisoformat(t)
    except ValueError:
        return None



RUN_WINDOW_WALL = None  # set below
WINDOWS = sorted({w["start"] for w in PHASE_WINDOWS.values()}
                 | {w["end"] for w in PHASE_WINDOWS.values()})
RUN_WINDOW_WALL = (WINDOWS[0], WINDOWS[-1])
RUN_WINDOW = RUN_WINDOW_WALL  # replaced below, once the offset is MEASURED
print(f"  the run touched windows {RUN_WINDOW[0]} .. {RUN_WINDOW[1]}  ({len(WINDOWS)} distinct)")
for name, w in sorted(PHASE_WINDOWS.items()):
    print(f"    phase {name}  {w['start']}  {w['seconds']:>6.1f}s  {w['cells']:>3} cells"
          + ("   STRADDLED" if w["straddled"] else ""))


# ---------------------------------------------------------------------------
# THE WINDOW OFFSET, MEASURED RATHER THAN ASSUMED.
#
# Run 1789366561 proved the report does NOT stamp a row with the window its
# records fell in: window_start 06:19:00Z carried min_record_time
# 06:19:30.587 and max 06:19:31.494 -- the whole burst landed AFTER the window
# closed, and it did so in 6 of 6 windows. A row labelled W holds the traffic
# of W+1, so every window-scoped query in this notebook was asking for the
# wrong rows, which is the entire reason Gate 2 went VOID twice and Gate 4
# reported a grant count from a window with no grants in it.
#
# The offset is DERIVED from the data, never hardcoded: it goes to zero on its
# own the day local-k8s fixes the attribution, and this cell then becomes a
# no-op that still proves it is zero. Nothing downstream has to be edited back.
_probe_lo, _probe_hi = osr.window_bounds(
    _rfc3339(RUN_WINDOW_WALL[0]).timestamp() - 3 * WINDOW_SECONDS, WINDOW_SECONDS)[0], \
    osr.window_bounds(_rfc3339(RUN_WINDOW_WALL[1]).timestamp() + 3 * WINDOW_SECONDS,
                      WINDOW_SECONDS)[0]
_probe = OS.report(osr.q_reports(
    report_type="summary", window=(_probe_lo, _probe_hi), size=200,
    schema_version=OS.schema_version,
    source=["window_start", "min_record_time", "max_record_time", "window_seconds"])).sources

_seen = collections.Counter()
for _r in _probe:
    _ws, _lo = _rfc3339(_r.get("window_start")), _rfc3339(_r.get("min_record_time"))
    if _ws is None or _lo is None:
        continue
    _secs = int(_r.get("window_seconds") or WINDOW_SECONDS)
    _lo_floor = _rfc3339(osr.window_bounds(_lo.timestamp(), _secs)[0])
    _seen[round((_ws - _lo_floor).total_seconds())] += 1

WINDOW_OFFSET_S, WINDOW_PROVENANCE = 0, ""
if not _seen:
    print("  WINDOW OFFSET: not measurable -- no summary row carried both timestamps. "
          "Queries run UNSHIFTED.")
elif len(_seen) > 1:
    # Disagreement is not something to average away: a per-row offset means the
    # attribution is not a constant lag and no single shift can be correct.
    print(f"  WINDOW OFFSET: INCONSISTENT across rows {dict(_seen)} -- refusing to guess. "
          "Queries run UNSHIFTED and every window-scoped gate below is suspect.")
    WINDOW_PROVENANCE = (f"Window attribution is INCONSISTENT across rows ({dict(_seen)}); no "
                         "single offset can correct it, so window-scoped gates were read "
                         "unshifted and must not be quoted.")
else:
    WINDOW_OFFSET_S = next(iter(_seen))
    print(f"  WINDOW OFFSET: {WINDOW_OFFSET_S:+d}s, agreed by all {sum(_seen.values())} "
          f"summary row(s) in range")
    if WINDOW_OFFSET_S:
        WINDOW_PROVENANCE = (
            f"The report stamps a row with a window {abs(WINDOW_OFFSET_S)}s "
            f"{'BEFORE' if WINDOW_OFFSET_S < 0 else 'AFTER'} the one its records fall in "
            f"(measured from {sum(_seen.values())} summary rows). Window-scoped queries here "
            "were shifted to compensate; the underlying defect is unfixed and the "
            "`record times fall INSIDE their own window` invariant still FAILS.")


def lbl(w):
    """The window LABEL that carries the traffic of wall-clock window `w`."""
    if not WINDOW_OFFSET_S or not w:
        return w
    return osr.window_bounds(_rfc3339(w).timestamp() + WINDOW_OFFSET_S, WINDOW_SECONDS)[0]


RUN_WINDOW = (lbl(RUN_WINDOW_WALL[0]), lbl(RUN_WINDOW_WALL[1]))
if RUN_WINDOW != RUN_WINDOW_WALL:
    print(f"  window-scoped queries use {RUN_WINDOW[0]} .. {RUN_WINDOW[1]} "
          f"(wall clock was {RUN_WINDOW_WALL[0]} .. {RUN_WINDOW_WALL[1]})")

_body = {"size": 0, "query": {"bool": {"filter": [
    {"prefix": {osr.kw("mdc.requestId"): f"nb-{RUN}-"}}]}}}
settled = OS.settle(_body, index=OS.logs_index, quiet_for=6, timeout=120, interval=3)
print(f"\n  records carrying this run's ids: {settled.total}")

FOUND = {}
for row in ROWS:
    rid = row["request_id"]
    if rid:
        FOUND[rid] = OS.for_request_id(rid)

_with_id = [r for r in ROWS if r["request_id"]]
_recovered = [r for r in _with_id if FOUND.get(r["request_id"])]
print(f"  correlation: {len(_recovered)}/{len(_with_id)} calls recovered by request id")

# THE SPLIT THAT MATTERS. A KEPT call must be findable and a miss there is a real
# fault; a COUNTED call is findable only incidentally, through whatever application
# lines it happened to emit. The unsplit ratio punishes the policy for working.
KEPT_TARGETS = {401, 403, 404, 409, 400, 500}
_kept = [r for r in _with_id if r["target"] in KEPT_TARGETS or (r["status"] or 0) >= 400]
_kept_found = [r for r in _kept if FOUND.get(r["request_id"])]
print(f"  of those, errors (rule 3 keeps ALL of them -- an ASSERTION): "
      f"{len(_kept_found)}/{len(_kept)}")
MISSING_ERRORS = [r["label"] for r in _kept if not FOUND.get(r["request_id"])]
if MISSING_ERRORS:
    print(f"  !! NOT FOUND, and rule 3 says they must be: {MISSING_ERRORS[:8]}")

  the run touched windows 2026-09-14T07:26:30Z .. 2026-09-14T07:29:30Z  (7 distinct)
    phase B  2026-09-14T07:26:30Z     1.0s   63 cells
    phase C  2026-09-14T07:27:00Z     1.0s  180 cells
    phase D  2026-09-14T07:27:30Z     0.5s   43 cells
    phase E  2026-09-14T07:28:00Z     0.1s    1 cells
    phase F  2026-09-14T07:28:30Z     0.0s    1 cells
    phase G  2026-09-14T07:29:00Z     0.1s    4 cells
    phase H  2026-09-14T07:29:30Z     0.6s   12 cells
  WINDOW OFFSET: INCONSISTENT across rows {-30: 5, 0: 1} -- refusing to guess. Queries run UNSHIFTED and every window-scoped gate below is suspect.

  records carrying this run's ids: 3336
  correlation: 286/286 calls recovered by request id
  of those, errors (rule 3 keeps ALL of them -- an ASSERTION): 235/235


## 12b -- The records themselves

Everything from here down aggregates. This prints the **documents**, one of each shape, so a
number that looks wrong can be checked against the record that produced it rather than against
another query -- which is the only way run `1789007773`'s eight "passing" gates were ever
understood, because no aggregate can tell you it matched nothing.

`summary`, `resource` and `principal` are three different documents in `polaris-report-*`, not
three views of one. The **audit** record is the other index entirely: `polaris-logs-*`, one
document per call, joined to the matrix by the `Polaris-Request-Id` this run stamped on every
request.

In [17]:
# RAW DOCUMENTS, EXACTLY AS OPENSEARCH HOLDS THEM.
#
# Everything below this cell aggregates: counts, sums, margins, verdicts. This
# prints the actual records, so a number that looks wrong can be checked
# against the document that produced it rather than against another query --
# which is how run 1789007773's eight "passing" gates were finally understood
# (they had matched nothing, and no aggregate can say that about itself).
#
# THREE SHAPES, and they are different documents, not three views of one:
#   summary   one per window per host  -- access_seen, parse_errors, carried_rows
#   resource  one per resource key per window -- requests/reads/writes/errors
#   principal one per principal per window -- the per-identity margin
#   audit     one per CALL, in polaris-logs-* -- the access log line itself
import json as _json


def _dump(label, result, n=1):
    print()
    print("=" * 78)
    print(f"  {label}   ({result.total} matching in the run's windows)")
    print("=" * 78)
    if not result.raw_sources:
        print("  NOTHING MATCHED. That is not an empty result to shrug at: every gate that")
        print("  reads this shape is VOID, and the cause is upstream of the gate.")
        return
    for _src in result.raw_sources[:n]:
        print(_json.dumps(_src, indent=2, ensure_ascii=False, sort_keys=True))


RAW = {}
for _type in ("summary", "resource", "principal"):
    RAW[_type] = OS.report(osr.q_reports(report_type=_type, window=RUN_WINDOW, size=5,
                                         schema_version=OS.schema_version))
    _dump(f"{_type.upper()} report document  (polaris-report-*, schema v{OS.schema_version})",
          RAW[_type])

# THE AUDIT SIDE. A different index, a different shipper, one document per call
# -- joined to the matrix by the Polaris-Request-Id this run stamped on every
# request. This is the record rule 3 promises to keep for every 4xx.
RAW["audit"] = OS.logs({"size": 5, "query": {"bool": {"filter": [
    {"prefix": {osr.kw("mdc.requestId"): f"nb-{RUN}-"}}]}}})
_dump("AUDIT record  (polaris-logs-*, one per call)", RAW["audit"])

# WHAT FIELDS ARE ACTUALLY PRESENT, per shape. The schema document is a claim;
# this is the answer. A field named here and missing above is the whole reason
# the oracle diff exists.
print()
for _k, _r in RAW.items():
    _keys = sorted({_f for _s in _r.raw_sources for _f in _s})
    print(f"  {_k:10} {len(_r.raw_sources)} doc(s), {len(_keys)} field(s): {_keys}")


  SUMMARY report document  (polaris-report-*, schema v3)   (6 matching in the run's windows)
{
  "@timestamp": "2026-09-14T07:29:33.674Z",
  "_msg": "polaris shipper report seq=191@benchmarks-fluent-bit-d94qc 2026-09-14T07:29:00Z..2026-09-14T07:29:30Z: 53 access lines, 46 kept, 7 counted (4 read, 3 POST), 25 errors kept (25 4xx, 0 5xx, 0 denied), 27 resources, 2 principals, 1 carried, 7847 bytes, 0 windows skipped",
  "_time": "2026-09-14T07:29:30Z",
  "access_counted": 7,
  "access_kept": 46,
  "access_seen": 53,
  "app": "polaris-shipper-report",
  "auth_denied": 0,
  "bytes_total": 7847,
  "carried_rows": 1,
  "counted_post": 3,
  "counted_read": 4,
  "distinct_principals": 2,
  "distinct_resources": 27,
  "errors_4xx": 25,
  "errors_5xx": 0,
  "errors_kept": 25,
  "hostname": "benchmarks-fluent-bit-d94qc",
  "level": "REPORT",
  "max_record_time": "2026-09-14T07:29:31.569180453Z",
  "min_record_time": "2026-09-14T07:29:30.594786271Z",
  "parse_errors": 0,
  "partial_window": "fals

## 13 -- The matrix

One row per cell. `target` is what the cell asked for, `status` is what it got, and a row where
they differ is **MISSED with the actual code** -- never recounted as coverage of whatever it hit.

In [18]:
import pandas as pd

M = pd.DataFrame(ROWS)
RESULTS = {(r["op_id"], r["target"]): {"verdict": r["verdict"], "actual": r["status"],
                                       "why": r["why"]} for r in ROWS}
COVERAGE = mx.coverage_by_operation(OPS, RESULTS)

print("  by target:")
for target in sorted({r["target"] for r in ROWS}, key=str):
    rows = [r for r in ROWS if r["target"] == target]
    cov = sum(1 for r in rows if r["verdict"] == mx.COVERED)
    err = sum(1 for r in rows if r["verdict"] == mx.ERROR)
    print(f"    {str(target):>4}  {cov:>3}/{len(rows):<3} covered"
          + (f"   {err} transport error(s)" if err else ""))

_full = [c for c in COVERAGE if not c["missed"] and not c["not_driven"]]
print(f"\n  operations with EVERY planned cell covered: {len(_full)}/{len(OPS)}")

_holes = [c for c in COVERAGE if c["missed"]]
print(f"  operations with a missed cell: {len(_holes)}")
for c in _holes[:25]:
    got = {t: RESULTS[(c['op_id'], t)]['actual'] for t in c['missed']}
    print(f"    {c['op_id']:34} missed {c['missed']}  got {got}")

pd.set_option("display.width", 200, "display.max_rows", 400, "display.max_colwidth", 44)
print("\n" + M[["op_id", "method", "target", "status", "verdict", "principal", "phase",
                "elapsed_ms"]].to_string(index=False))

  by target:
       2   50/63  covered
     400   16/28  covered
     401   59/63  covered
     403   42/62  covered
     404   54/55  covered
     409    7/15  covered

  operations with EVERY planned cell covered: 30/63
  operations with a missed cell: 33
    getConfig                          missed [403]  got {403: 200}
    getToken                           missed [400]  got {400: 500}
    createNamespace                    missed [400]  got {400: 500}
    dropNamespace                      missed [403, 409]  got {403: 404, 409: 404}
    registerTable                      missed [2, 409]  got {2: 400, 409: 400}
    createTable                        missed [409]  got {409: 403}
    dropTable                          missed [403]  got {403: 404}
    updateTable                        missed [400, 409]  got {400: 200, 409: 200}
    reportMetrics                      missed [403, 404]  got {403: 204, 404: 204}
    planTableScan                      missed [2, 400, 401, 403]  got {2: 

## 14 -- Gates 1 to 6

Verbatim from `GUIDE-schema-v3-testing.md`, through `os_report` so the schema filter, the
`value_count` pairing and the `.keyword` suffixes cannot be forgotten at a call site.

Each prints PASS / FAIL / **VOID**. VOID is not a soft FAIL: it means the window carried nothing
that could answer the question, and calling that a pass invents a measurement while calling it a
failure invents a fault.

In [19]:
GATES = {}


def verdict(name, ok, detail=""):
    GATES[name] = {"ok": ok, "detail": detail}
    mark = {True: "PASS", False: "FAIL", None: "VOID"}[ok]
    print(f"  {name:34} {mark}   {detail}")


# HOW MANY ROWS ARE EVEN IN SCOPE. Run 1789007773 answered PASS for two gates
# that had looked at nothing: "Gate 6 -- only the synthetic keys came back" and
# "Gate 4 grants fold into the role -- no separate grants row" were both true
# of an EMPTY result set, because window_start.keyword did not exist and every
# window-scoped query matched nothing. A gate that cannot tell "nothing is
# wrong" from "I found nothing" is the exact defect this notebook exists to
# catch, and it was in the notebook.
SCOPE = OS.report(osr.q_reports(window=RUN_WINDOW, size=0,
                                schema_version=OS.schema_version)).total
print(f"  {SCOPE} v{OS.schema_version} report row(s) in the run's windows "
      f"({RUN_WINDOW[0]} .. {RUN_WINDOW[1]})")
if not SCOPE:
    print("  !! NOTHING IS IN SCOPE. Every gate below is VOID -- not FAIL -- and the cause is")
    print("     upstream of them: either no report row was written for these windows, or the")
    print("     field names resolved at preflight do not match the index. Check the `fields`")
    print("     line in cell 0c before reading a single verdict here.")

# -- Gate 1: the margin invariants ---------------------------------------
MARGINS = OS.margins(RUN_WINDOW)
_answered = [w for w in MARGINS if w["ok"] is not None]
_bad = [w for w in _answered if not w["ok"]]
verdict("Gate 1 margins",
        None if not _answered else not _bad,
        f"{len(_answered)}/{len(MARGINS)} windows answered"
        + ("" if not _bad else "; " + "; ".join(f"{w['window_start']}: {w['why']}" for w in _bad[:3])))

# -- Gate 0b: the array split, on THIS side ------------------------------
_split = OS.report_types(window=RUN_WINDOW)
# VOID, not FAIL, when the window held no report rows at all: a window with
# access_seen == 0 has no resource or principal rows to split INTO, and calling
# that a failure invents a pipeline fault.
_split_ok = None if not _split else sorted(_split) == ["principal", "resource", "summary"]
verdict("Gate 0b array split", _split_ok, str(_split))

# -- Gate 2: last_write_bytes, THE FEATURE -------------------------------
_claim = OS.report(osr.q_last_write(resource=None, kind="table",
                                    window=(lbl(GATE2["window"]), lbl(GATE2["window"])))).sources
_records = OS.logs(osr.q_access_records(GATE2["window"], GATE2["window_end"],
                                        method="POST", path_wildcard="*/tables/*")).sources
_2xx = [r for r in _records if 200 <= int(r.get("http_status", 0)) < 300]

# WHICH OF THE TWO IS IT? "no table row carried last_write_bytes" was the whole
# of this gate's output in run 1789008899, and it cannot distinguish `the
# filter never set the field` from `there is no table row here at all` -- which
# have opposite remedies. Print the rows.
_tables = OS.report(osr.q_reports(report_type="resource",
                                  window=(lbl(GATE2["window"]), lbl(GATE2["window"])), size=50,
                                  schema_version=OS.schema_version,
                                  source=["resource", "resource_kind", "requests", "reads",
                                          "writes", "last_write_bytes", "last_read_bytes",
                                          "response_bytes"])).sources
_table_rows = [r for r in _tables if r.get("resource_kind") == "table"]
print(f"  Gate 2 scope: {len(_tables)} resource row(s) in {lbl(GATE2['window'])}, "
      f"{len(_table_rows)} of kind `table`")
for _r in _table_rows[:6]:
    print(f"    {str(_r.get('resource'))[-52:]:52} writes={_r.get('writes')} "
          f"last_write_bytes={_r.get('last_write_bytes')!r}")
if not _claim:
    verdict("Gate 2 last_write_bytes", None,
            (f"the commit itself returned {GATE2['status']}" if GATE2["status"] >= 400
             else (f"{len(_table_rows)} table row(s) in this window and none carries "
                   "last_write_bytes. THIS SAYS NOTHING ABOUT WHETHER THE FEATURE WORKS: "
                   "the OpenSearch export of 2026-09-14 shows last_write_bytes and "
                   "last_read_bytes firing wherever a sized write or read occurred "
                   "(last write 536 / 117, last read 53) and absent exactly where the "
                   "response was 0 bytes, which is the `> 0` guard working. A VOID here "
                   "means this window held no sized table write -- check WHICH window was "
                   "read before blaming the filter"
                   if _table_rows else
                   "no table row at all in the commit's window -- the commit did not reach "
                   "the filter, or it was counted into a different window")))
elif not _2xx:
    verdict("Gate 2 last_write_bytes", None,
            "the report claims a value and the access log has no 2xx POST to compare it to")
else:
    _last = sorted(_2xx, key=lambda r: r.get("@timestamp"))[-1]
    _want = int(_last.get("response_size") or 0)
    _got = int(_claim[0].get("last_write_bytes") or 0)
    verdict("Gate 2 last_write_bytes", _got == _want,
            f"report={_got} last 2xx POST response_size={_want} on {_claim[0].get('resource')}")

# -- Gate 2, the negative half -------------------------------------------
_absent = OS.report(osr.q_absent_last_write(GATE2_NEG["table"],
                                            (GATE2_NEG["window"], GATE2_NEG["window"])))
_present = OS.report(osr.q_last_write(resource=GATE2_NEG["table"], kind=None,
                                      window=(lbl(GATE2_NEG["window"]), lbl(GATE2_NEG["window"]))))
verdict("Gate 2 delete leaves it ABSENT",
        None if not (_absent.total or _present.total) else _present.total == 0,
        f"rows without the field={_absent.total}  rows WITH it={_present.total} (want 0)")

# -- Gate 3: classification ----------------------------------------------
_cls = OS.report(osr.q_classification(window=RUN_WINDOW)).aggs
_pairs = {b["key"]: [k["key"] for k in b["kind"]["buckets"]]
          for b in (_cls.get("api") or {}).get("buckets", [])}
_kinds = {k for v in _pairs.values() for k in v}
verdict("Gate 3 classification",
        None if not _pairs else ("management" not in _kinds),
        f"{_pairs}" + ("   resource_kind must never read 'management'"
                       if "management" in _kinds else ""))

# -- Gate 4: roles and the privilege count -------------------------------
_roles = OS.report(osr.q_role_rows(kind="catalog-role",
                                   window=(lbl(GATE4["window"]), lbl(GATE4["window"])))).sources
# EXACT, NOT SUBSTRING. `mx_<run>_crole` is a prefix of `mx_<run>_crole2`
# and of `mx_<run>_crole_doomed`, so `in` matched all three and `_mine[0]`
# was whichever came back first. Run 1789365275 reported `writes=1
# granted=3` against `_crole_doomed`, a role that received no grants, and
# that sentence went into REPORT-for-local-k8s.md as a work item.
_mine = [r for r in _roles
         if str(r.get("resource", "")).endswith("/" + GATE4["role"])]
# EVERY catalog-role row in scope. Run 1789008899 printed `writes=1 granted=3`
# off _mine[0] and gave no way to tell an undercounting filter from a gate
# reading the wrong row -- so the finding could not be sent anywhere.
print(f"  Gate 4 scope: {len(_roles)} catalog-role row(s) in {lbl(GATE4['window'])}"
      + (f"  (wall clock {GATE4['window']}, offset {WINDOW_OFFSET_S:+d}s)"
         if WINDOW_OFFSET_S else ""))
for _r in _roles[:8]:
    print(f"    {str(_r.get('resource'))[-56:]:56} req={_r.get('requests')} "
          f"reads={_r.get('reads')} writes={_r.get('writes')} "
          f"denied={_r.get('auth_denied')}")
print(f"    driven: {len(GATE4['granted'])} grant(s) "
      f"{[g['status'] for g in GATE4['granted']]}, denial {GATE4['denied_status']}")
if not _mine:
    verdict("Gate 4 privilege count", None,
            f"no catalog-role row for {GATE4['role']} in {GATE4['window']}; "
            f"rows present: {[str(r.get('resource'))[-40:] for r in _roles[:4]]}")
else:
    _w = int(_mine[0].get("writes") or 0)
    verdict("Gate 4 privilege count", _w == GATE4["expected_writes"],
            f"writes={_w} granted={GATE4['expected_writes']} on {_mine[0].get('resource')}")
    _denied = int(_mine[0].get("auth_denied") or 0)
    verdict("Gate 4 denial forces the row", _denied >= 1,
            f"auth_denied={_denied} (the ROLE_KINDS exemption; 0 means the 403 fell to __errors__)")

_grants_row = [r for r in _roles if str(r.get("resource", "")).endswith("/grants")]
verdict("Gate 4 grants fold into the role",
        None if not _roles else not _grants_row,
        "a .../grants row of its own means the fold broke" if _grants_row
        else ("no catalog-role row in scope at all -- nothing was examined"
              if not _roles else "no separate grants row"))

# -- Gate 5: the two synthetic buckets -----------------------------------
_syn = OS.synthetic_split(window=RUN_WINDOW)
verdict("Gate 5 __errors__ holds only errors", _syn["verdict"], _syn["why"] or
        f"requests={_syn['errors']['requests']!r}")
_other_req = _syn["other"]["requests"]
verdict("Gate 5 __other__ is overflow only",
        None if _other_req.absent else _other_req.as_int() == 0,
        f"__other__ requests={_other_req!r} -- non-zero means a window really carried 500+ "
        "distinct resources, or the split regressed")

# -- Gate 6: what still needs a classification rule ----------------------
UNCLASSIFIED = OS.unclassified(window=RUN_WINDOW)
_resource_rows = OS.report(osr.q_reports(report_type="resource", window=RUN_WINDOW, size=0,
                                         schema_version=OS.schema_version)).total
verdict("Gate 6 every path has a rule",
        None if not _resource_rows else not UNCLASSIFIED,
        (f"{len(UNCLASSIFIED)} real path(s) with no RESOURCE_PATTERNS entry" if UNCLASSIFIED
         else ("no resource row in scope -- nothing was examined" if not _resource_rows
               else f"only the synthetic keys came back, over {_resource_rows} resource row(s)")))
for u in UNCLASSIFIED[:20]:
    print(f"      {u['api_kind']:12} {u['resource']}  ({u['rows']} row(s))")

  159 v3 report row(s) in the run's windows (2026-09-14T07:26:30Z .. 2026-09-14T07:29:30Z)
  Gate 1 margins                     PASS   6/6 windows answered
  Gate 0b array split                PASS   {'resource': 135, 'principal': 18, 'summary': 6}
  Gate 2 scope: 2 resource row(s) in 2026-09-14T07:28:00Z, 2 of kind `table`
    at/namespaces/probe_ns/tables/mx_1789370776_droponly writes=1 last_write_bytes=None
    x1789370776_cat/namespaces/probe_ns/tables/probe_tbl writes=0 last_write_bytes=None
  Gate 2 last_write_bytes            VOID   2 table row(s) in this window and none carries last_write_bytes. THIS SAYS NOTHING ABOUT WHETHER THE FEATURE WORKS: the OpenSearch export of 2026-09-14 shows last_write_bytes and last_read_bytes firing wherever a sized write or read occurred (last write 536 / 117, last read 53) and absent exactly where the response was 0 bytes, which is the `> 0` guard working. A VOID here means this window held no sized table write -- check WHICH window was read bef

## 15 -- The schema's own invariants

From `SCHEMA-report.md`, *Invariants worth asserting*. The fourth one --
`max_record_time - min_record_time <= window_seconds` -- is checked **client-side**: the fields
are RFC3339 strings and parse fine here, whatever OpenSearch typed them as. What the `text`
mapping kills is date maths *inside* the index, not the invariant.

In [20]:
INVARIANTS = []


def inv(name, ok, detail=""):
    INVARIANTS.append({"name": name, "ok": ok, "detail": detail})
    mark = {True: "PASS", False: "FAIL", None: "VOID"}[ok]
    print(f"  {name:44} {mark}   {detail}")


for w in MARGINS:
    if w["ok"] is None:
        continue
    inv(f"margin {w['window_start']}", w["ok"], w["why"] or
        f"{w['resource_requests'].as_int(0)} == {w['expected']}")

# carried_rows == the number of rows with requests == 0
_zero = OS.report({"size": 0, "query": {"bool": {"filter":
        osr._filters({"range": {"requests": {"lte": 0}}},
                     schema_version=OS.schema_version, window=RUN_WINDOW)}}})
_carried = [w["carried_rows"] for w in MARGINS if not w["carried_rows"].absent]
inv("carried_rows counts the zero-request rows",
    None if not _carried else True,
    f"rows with requests<=0: {_zero.total}; summary carried_rows total: "
    f"{sum(m.as_int(0) for m in _carried)}  -- v3 merges resources AND principals into one "
    "counter (review #3), so this is reported, not asserted")

# report_seq: contiguous per (hostname, schema_version)
_seqs = OS.report(osr.q_reports(report_type="summary", window=RUN_WINDOW, size=500,
                                source=["report_seq", "hostname", "window_start"])).sources
_by_host = {}
for r in _seqs:
    _by_host.setdefault(r.get("hostname"), []).append(int(r.get("report_seq") or 0))
for host, seqs in _by_host.items():
    seqs = sorted(seqs)
    contiguous = seqs and (max(seqs) - min(seqs) + 1 == len(set(seqs)))
    inv(f"report_seq contiguous on {host}", bool(contiguous),
        f"{min(seqs)}..{max(seqs)} over {len(set(seqs))} row(s)"
        + ("" if contiguous else "   A GAP: a window closed with no tick, or the pod restarted"))

# min/max record time, parsed client-side
import datetime as _dt

_rt = OS.report(osr.q_reports(report_type="summary", window=RUN_WINDOW, size=200,
                              source=["min_record_time", "max_record_time", "window_start",
                                      "window_seconds"])).sources
_checked, _bad, _unparsed = 0, [], []
_outside = []
for r in _rt:
    lo, hi = _rfc3339(r.get("min_record_time")), _rfc3339(r.get("max_record_time"))
    if r.get("min_record_time") and lo is None:
        _unparsed.append(r.get("min_record_time"))
    if not lo or not hi:
        continue
    _checked += 1
    _secs = int(r.get("window_seconds") or WINDOW_SECONDS)
    if (hi - lo).total_seconds() > _secs:
        _bad.append((r.get("window_start"), (hi - lo).total_seconds()))
    # THE CHECK THE SPAN TEST CANNOT MAKE. Run 1789365275 carried
    # window 05:57:30..05:58:00 with min_record_time 05:58:00.607 and
    # max 05:58:01.475 -- BOTH AFTER THE WINDOW CLOSED -- and the span
    # check passed on 0.87s without noticing. A window is attributed by
    # when Fluent Bit PROCESSED the record, not by when it happened, and
    # every per-window measurement carries that offset.
    _ws = _rfc3339(r.get("window_start"))
    if _ws is not None:
        _we = _ws + _dt.timedelta(seconds=_secs)
        if lo < _ws or hi > _we:
            _outside.append({"window_start": r.get("window_start"),
                             "min": r.get("min_record_time"), "max": r.get("max_record_time"),
                             "lead_s": round((_ws - lo).total_seconds(), 3),
                             "lag_s": round((hi - _we).total_seconds(), 3)})

inv("record times fall INSIDE their own window", None if not _checked else not _outside,
    f"{_checked} window(s) checked; {len(_outside)} carried a timestamp outside "
    f"[window_start, window_start+window_seconds]"
    + (f"; worst lag {max(o['lag_s'] for o in _outside)}s, e.g. {_outside[0]}"
       if _outside else "")
    + "   -- outside means the window is assigned by PROCESSING time, not record time")
# THE SIGNAL THE FINDINGS CELL GATES ON. Derived from the invariant that just
# ran, never from WINDOW_OFFSET_S -- which is 0 both when there is NO offset and
# when the offset could not be determined, so gating on it failed OPEN in exactly
# the case it was built for, and sent two unreproduced items to local-k8s for a
# third run.
WINDOWS_TRUSTED = bool(_checked) and not _outside

if _unparsed:
    inv("every record timestamp parses", False,
        f"{len(_unparsed)} unparseable, e.g. {_unparsed[:2]}")
inv("record span fits inside the window", None if not _checked else not _bad,
    f"{_checked} window(s) carried both timestamps"
    + (f"; OVER: {_bad[:3]}" if _bad else "")
    + ("   (idle windows write \"\" and are skipped -- that is review #2, not a pass)"
       if _checked < len(_rt) else ""))

  margin 1789370790000                         PASS   180 == 180
  margin 1789370820000                         PASS   44 == 44
  margin 1789370850000                         PASS   1 == 1
  margin 1789370880000                         PASS   1 == 1
  margin 1789370910000                         PASS   7 == 7
  margin 1789370940000                         PASS   53 == 53
  carried_rows counts the zero-request rows    PASS   rows with requests<=0: 49; summary carried_rows total: 49  -- v3 merges resources AND principals into one counter (review #3), so this is reported, not asserted
  report_seq contiguous on benchmarks-fluent-bit-d94qc PASS   186..191 over 6 row(s)
  record times fall INSIDE their own window    FAIL   6 window(s) checked; 6 carried a timestamp outside [window_start, window_start+window_seconds]; worst lag 1.569s, e.g. {'window_start': '2026-09-14T07:29:00Z', 'min': '2026-09-14T07:29:30.594786271Z', 'max': '2026-09-14T07:29:31.569180453Z', 'lead_s': -30.595, 'lag_s': 1.

## 16 -- Findings, and the report for `local-k8s`

Two documents, written by this cell.

`doc-api-status-matrix-results.md` is the run of record for **this** repo. `REPORT-for-local-k8s.md`
is the work list for the **pipeline** repo: only the items whose fix lives there -- a
classification rule, an index template, a gate that cannot fail -- each with what was measured
and what to change.

Test-driven, as agreed: the run does not fix the pipeline, it produces the list.

In [ ]:
# EVERY 500 IN THE RUN, not only the ladder's. Run 1789007773 printed "500
# ladder: NOT PROVOKED" -- true of the four storage rungs it walks -- while
# FIVE 500s sat in the matrix unremarked, provoked by malformed request bodies
# on getToken, createNamespace, renameTable, renameView and updateCatalog. A
# true sentence that hides the finding is the failure mode this repo keeps
# paying for, so the accounting reads all 286 rows.
ALL_500 = [r for r in ROWS if (r["status"] or 0) >= 500]
FIVE_HUNDRED["matrix_500"] = ALL_500
print(f"  500s in the whole run: {len(ALL_500)}   (ladder: "
      f"{FIVE_HUNDRED['winner'] or 'NOT PROVOKED'})")
for r in ALL_500:
    print(f"    {r['op_id']:24} target {r['target']:>4} -> {r['status']}  "
          f"as {r['principal']}  phase {r['phase']}")

# HANDLED OR UNHANDLED? A 500 that Polaris caught and mapped carries no
# throwable, and reporting that as a lost stack trace is the wrong finding --
# lc.trace_verdict separates the three cases and only `absent` accuses the
# pipeline (rule 2 keeps every non-access-log record whatever its level).
TRACE, TRACE_DETAIL = {}, {}
for r in ALL_500:
    _recs = FOUND.get(r["request_id"]) or []
    _pair = lc.error_record_pair(_recs, r["request_id"])
    TRACE[r["label"]] = lc.trace_verdict(_pair)
    TRACE_DETAIL[r["label"]] = {"records": len(_recs), "matched": _pair["access_records"]
                                + _pair["app_records"], "loggers": _pair["loggers"],
                                "levels": _pair["levels"],
                                "exception_fields": _pair["exception_fields"]}
if TRACE:
    print("  trace verdicts:", {k.split("@")[0]: v for k, v in TRACE.items()})
    for _k, _d in list(TRACE_DETAIL.items())[:4]:
        print(f"    {_k.split('@')[0]:22} records={_d['records']} matched={_d['matched']} "
              f"levels={_d['levels']} loggers={_d['loggers'][:2]}")

# EVIDENCE ABOUT THE RECORD SHAPE ITSELF. Run 1789008899 returned `absent` for
# all eight 500s -- no application line, which rule 2 forbids and which would be
# a SERIOUS pipeline finding. The likelier cause is a dict access:
# error_record_pair filters on r.get("mdc.requestId"), VictoriaLogs FLATTENS
# nested JSON on ingest and OpenSearch does not, and a query addresses the
# nested field either way -- which is why correlation reported 286/286 and hid
# it. os_report.Result.sources now flattens; this prints the raw shape so the
# claim is settled by looking rather than by reasoning.
if ALL_500:
    _rid = ALL_500[0]["request_id"]
    _raw = OS.logs({"size": 2, "query": {"bool": {"filter": [
        {"term": {osr.kw("mdc.requestId"): _rid}}]}}})
    print()
    print(f"  record shape for {_rid}:")
    for _r in _raw.raw_sources[:1]:
        print(f"    raw _source keys      {sorted(_r)[:12]}")
    for _r in _raw.sources[:1]:
        print(f"    flattened keys        {sorted(_r)[:12]}")
        print(f"    mdc.requestId reads   {_r.get('mdc.requestId')!r}")

# A 500 WHOSE RECORDS CARRY NO ERROR LINE, YET VERDICT `unhandled`.
# Run 1789366561: createView returned 500, 50 records matched, levels were
# ['DEBUG', 'INFO'] and nothing else. Either trace_verdict is wrong about what
# it saw, or rule 2 lost the ERROR record -- OPPOSITE remedies, and the run
# cannot tell them apart, so it must say so rather than pick one.
TRACE_CONTRADICTIONS = [
    {"label": _k, "verdict": TRACE[_k], "levels": _d["levels"], "records": _d["records"]}
    for _k, _d in TRACE_DETAIL.items()
    if TRACE.get(_k) == "unhandled" and "ERROR" not in (_d.get("levels") or [])
]
if TRACE_CONTRADICTIONS:
    print()
    print(f"  !! {len(TRACE_CONTRADICTIONS)} 500(s) verdict `unhandled` with NO ERROR record:")
    for _c in TRACE_CONTRADICTIONS:
        print(f"     {_c['label'].split('@')[0]:22} records={_c['records']} levels={_c['levels']}")

FINDINGS = []


def finding(where, title, measured, remedy, severity="finding"):
    FINDINGS.append({"where": where, "title": title, "measured": measured,
                     "remedy": remedy, "severity": severity})


# THE INVARIANTS WERE NEVER READ HERE. This cell built findings from GATES alone,
# so the one defect this notebook has proven on three consecutive runs -- the
# window attribution -- reached NO document at all, while two unreproduced gate
# results did. An instrument whose output never leaves the notebook is not an
# instrument.
INVARIANT_REMEDIES = {
    "record times fall INSIDE their own window": (
        "a report row must be stamped with the window its records fall in. The Measured line "
        "above is THIS run; what follows is what earlier runs and an OpenSearch export add, "
        "cited separately so the figures are not mistaken for one another. CORROBORATION, run "
        "1789368559: seq=117 is labelled 06:52:00..06:52:30 and the kept access-log lines in it "
        "are stamped 06:52:31 -- visible in OpenSearch without the notebook. RECONCILIATION, "
        "run 1789370776: every phase this notebook drives appears one window EARLY in the label "
        "(phase C's 180 calls under 07:26:30, phase D's 44 under 07:27:00, and so on). SHAPE, "
        "and this is why no downstream shift can correct it: the error is not a constant. "
        "Measured across rows it is {-30: 5, 0: 1}, and the single 0 is the fixture-setup "
        "window, whose traffic is spread through the window rather than arriving as a burst "
        "just after the boundary. The window a row gets therefore depends on WHEN INSIDE IT "
        "the traffic landed, which no consumer can correct for. Every per-window number here, "
        "and every window-scoped gate, is read from the wrong row until this is fixed at the "
        "source."),
}
for _inv in INVARIANTS:
    if _inv["ok"] is False:
        finding("local-k8s", f"invariant FAILED: {_inv['name']}", _inv["detail"],
                INVARIANT_REMEDIES.get(_inv["name"],
                                       "see SCHEMA-report.md, 'Invariants worth asserting'"),
                severity="SERIOUS")

# 0. THE FILTER'S OWN UNIT TEST. Cell 4 has always built a GATE7 dict and
# NOTHING has ever read it -- that cell's own text promised "the findings cell
# records that the filter's own unit test never executed", and no cell did. A
# scoped FAIL is a pipeline defect the run deliberately continued past, so it
# has to surface here or continuing past it becomes hiding it.
if GATE7["ran"] and not GATE7["ok"]:
    _g7_remedy = ("every failure line is in the run log above; fix the deployed filter, then "
                  "re-run this notebook from the top.")
    if any("record_time" in _ln for _ln in GATE7["failures"]):
        _g7_remedy = (
            "the schema requires min_record_time / max_record_time ABSENT on an idle window "
            'and the filter emits "". Emit nil. HYPOTHESIS, not yet settled: this is also the '
            "CAUSE of the text mapping in the index-mapping finding below -- OpenSearch types "
            "a field from the first document it indexes, the first window of a day is almost "
            'always idle at 30s windows, and "" is not a parseable date. If that holds, the '
            "index template is a workaround and this is the fix. Settle it by reading "
            "min_record_time out of the OLDEST polaris-report-* index and checking whether the "
            'earliest document carries "".')
    finding("local-k8s", "Gate 7: the deployed Lua fails its own unit test",
            f"{len(GATE7['failures'])} assertion(s) of {GATE7['assertions']} failed against "
            f"{GATE7['source']} -- " + "; ".join(GATE7["failures"][:6]),
            _g7_remedy,
            severity="SERIOUS" if GATE7["unscoped"] else "finding")
elif not GATE7["ran"]:
    finding("this repo", "Gate 7 never executed -- the filter is UNPROVEN, not passed",
            GATE7["detail"] or "no detail recorded",
            "no gate failure below can be attributed to the query rather than to the filter "
            "until this runs. It needs lua5.4 and the local-k8s checkout on the Jupyter host.")

# 1. Endpoints with no classification rule -- the reason the whole grid exists.
if UNCLASSIFIED:
    finding("local-k8s", "endpoints with no RESOURCE_PATTERNS rule",
            "; ".join(f"{u['api_kind']} {u['resource']}" for u in UNCLASSIFIED[:12]),
            "add a RESOURCE_PATTERNS entry per path, then re-run Gate 6. These are visible "
            "ONLY because the grid drove all 63 operations -- Gate 6 has never had traffic on "
            "them before.")

# 2. The mapping traps.
for note in MAPPING_NOTES:
    finding("local-k8s", "index mapping", note,
            "an index template for polaris-report-* typing min_record_time / max_record_time as "
            "date (or date_nanos), and the numeric report fields as long. LIKELY CAUSE, REVISED "
            "2026-09-14 -- not the empty string the earlier note blamed: the same index maps "
            "window_start as `date`, and window_start carries NO fractional part while "
            "min/max_record_time carry NINE digits (...30.587684196Z). Dynamic date detection "
            "declines that, and the field is text for the life of the index. So the options "
            "are an explicit mapping or truncating to milliseconds in the filter, and a "
            "template alone leaves every future index one such field away from the same "
            "thing. Decisive free test: the next day's index is written entirely by the "
            "rolled filter.")

# 3. Gates that cannot fail as written.
finding("guide", "Gate 5's term filter matches nothing as written",
        'GUIDE-schema-v3-testing.md Gate 5 uses {"term": {"resource": "__errors__"}}. On a '
        "dynamic-mapped index that runs against the analysed text field, where the standard "
        "analyser has already turned __errors__ into the token `errors`.",
        "use resource.keyword (Gate 2's resource_kind filter has the same shape). Until then "
        "that gate reports zero and passes without having looked.")

# Remedies for the gates that have a KNOWN fix. "See the guide's stated failure
# mode" is not a work item for another repo, and run 1789365275 shipped two of
# them to local-k8s that way.
GATE_REMEDIES = {
    "Gate 4 privilege count": (
        "a grant PUT on .../catalog-roles/<role>/grants must fold into the <role> row and "
        "increment `writes` once per grant. Measured N grants -> writes=1 means the fold is "
        "keeping the LAST grant rather than summing them, or the role span is being rematched "
        "per request. Check `classify`'s returned span for the /grants suffix."),
    "Gate 4 denial forces the row": (
        "a 403 on a catalog-role with no successful request to that role in the same window "
        "must STILL create the role's row with auth_denied=1 and increment role_keys_forced on "
        "the summary. auth_denied=0 with the 403 in __errors__ means the ROLE_KINDS exemption "
        "to the `create=false` guard is not firing for catalog-role."),
    "Gate 2 last_write_bytes": (
        "last_write_bytes must carry the response_size of the last 2xx write in the window. "
        "Check the notebook's window attribution first -- if the record times fall outside "
        "their window (see the invariants), the gate may be reading a window the commit never "
        "landed in, and that is a harness fault, not a filter one."),
}

# 4. Whatever the gates actually found.
for name, g in GATES.items():
    if g["ok"] is False:
        # A WINDOW-SCOPED GATE CANNOT ACCUSE THE PIPELINE WHILE THE WINDOW
        # ITSELF IS IN DOUBT. Run 1789366561 shipped two Gate 4 items to
        # local-k8s that were measured in a window containing no grants.
        # While the offset is non-zero the finding stays HERE, named as ours,
        # until a run with the shift applied reproduces it.
        _windowed = name.startswith("Gate 2") or name.startswith("Gate 4")
        if _windowed and not WINDOWS_TRUSTED:
            finding("this repo", f"{name} FAILED -- measured under a window offset",
                    g["detail"] + "   [the window-attribution invariant FAILED this run, so this "
                    "gate was read from a row that may not hold its traffic; NOT "
                    "reproduced on a pipeline whose windows are trustworthy]",
                    "re-run and confirm before sending anywhere: " + GATE_REMEDIES.get(name, ""))
        else:
            finding("local-k8s", f"{name} FAILED", g["detail"],
                    GATE_REMEDIES.get(name, "see the gate's stated failure mode in "
                                            "GUIDE-schema-v3-testing.md"))
    elif g["ok"] is None:
        finding("this repo", f"{name} VOID", g["detail"],
                GATE_REMEDIES.get(name, "the window carried nothing that could answer "
                                        "it -- re-drive, do not re-read"))

if TRACE_CONTRADICTIONS:
    finding("undecided", "a 500 was judged `unhandled` with no ERROR record in its trace",
            "; ".join(f"{c['label'].split('@')[0]} ({c['records']} records, levels "
                      f"{c['levels']})" for c in TRACE_CONTRADICTIONS),
            "TWO candidate causes with opposite fixes and this run cannot separate them: "
            "(a) `lc.trace_verdict` calls it unhandled off something other than an ERROR "
            "line, in which case the verdict is wrong and no pipeline change is needed; or "
            "(b) rule 2 lost the ERROR record, which it forbids, and that is a SERIOUS "
            "retention gap. Settle it by reading the raw records for one such request id and "
            "checking whether an ERROR line exists in polaris.log at all.",
            severity="SERIOUS")

# THE 403 -> 404 BLOCK IS NOT A MISS, IT IS A LEDGER ENTRY. Polaris refuses to
# confirm a resource exists to a principal that may not see it, so the denied
# identity gets 404 where the matrix asked for 403. Counting these as misses
# has made the coverage figure look like a Polaris gap in every run so far.
_hidden = sorted({r["op_id"] for r in ROWS
                  if r["target"] == 403 and r["status"] == 404 and r["verdict"] == mx.MISSED})
if _hidden:
    finding("this repo", f"{len(_hidden)} cell(s) targeting 403 returned 404 -- existence hiding",
            ", ".join(_hidden[:14]),
            "this is Polaris declining to confirm existence to an unauthorised principal, "
            "not a coverage gap. It belongs in the not-reachable ledger with that reason, so "
            "the 403 column stops reading as a hole. Until it is in the ledger it will be "
            "re-found every run.")

# COVERAGE IS NOT REPRODUCIBLE WHILE THE PG-HA 500s FIRE. Run 1789365275 saw 4
# 500s and 228 covered; run 1789366561 saw 8 and 225, off the SAME code. Two
# runs cannot be compared cell-by-cell until this is said out loud.
if ALL_500:
    finding("this repo", f"coverage is not reproducible run to run: {len(ALL_500)} "
            "nondeterministic 500(s)",
            "; ".join(sorted({f"{r['op_id']}@{r['target']}" for r in ALL_500})),
            "these are the PG-HA read-after-write signature, not harness faults, and they "
            "land on different cells each run -- so the covered/missed totals move without "
            "any code changing. Quote coverage with the 500 count beside it, never alone.")

# 5. Cells that missed, grouped by what they hit instead.
_miss_by = {}
for r in ROWS:
    if r["verdict"] == mx.MISSED:
        _miss_by.setdefault((r["target"], r["status"]), []).append(r["op_id"])
for (target, actual), ops_ in sorted(_miss_by.items(), key=str):
    finding("this repo", f"{len(ops_)} cell(s) targeting {target} returned {actual}",
            ", ".join(ops_[:10]), "either the driver is wrong or this build does not produce "
            f"{target} on those endpoints -- decide which, and record it in the ledger")

# 5b. THE 500s, wherever they came from.
if ALL_500:
    _unhandled = [k for k, v in TRACE.items() if v == "unhandled"]
    _absent = [k for k, v in TRACE.items() if v == "absent"]
    finding("polaris", f"{len(ALL_500)} response(s) of 500 in this run",
            "; ".join(f"{r['op_id']} (target {r['target']}, as {r['principal']})"
                      for r in ALL_500)
            + f"   trace verdicts: {sorted(set(TRACE.values()))}",
            "these are reachable through the API alone, with no cluster surgery -- which "
            "answers the standing question in .memory/roadmap.md about provoking an "
            "unhandled exception. `errors_5xx` can now be exercised on demand."
            + (f" UNHANDLED (a throwable survived): {_unhandled}." if _unhandled else "")
            + (f" NO application line at all for {_absent}, which rule 2 forbids."
               if _absent else ""),
            severity="SERIOUS" if _absent else "finding")

# 6. Errors that rule 3 promises to keep and did not.
if MISSING_ERRORS:
    finding("local-k8s", "rule 3 kept fewer errors than it promises",
            f"{len(MISSING_ERRORS)} error call(s) left no record: {MISSING_ERRORS[:8]}",
            "rule 3 keeps every http_status >= 400 with no cap. A missing one is a real gap "
            "in the retention policy or in the shipper's delivery.", severity="SERIOUS")

print(f"  {len(FINDINGS)} finding(s)")
for f in FINDINGS:
    print(f"    [{f['where']:>10}] {f['title']}")

STAMP = datetime.now(timezone.utc).strftime("%Y-%m-%d %H:%MZ")


def _table(rows, cols):
    out = ["| " + " | ".join(cols) + " |", "|" + "|".join("---" for _ in cols) + "|"]
    for r in rows:
        out.append("| " + " | ".join(str(r.get(c, "")) for c in cols) + " |")
    return "\n".join(out)


results = [
    f"# API status matrix -- run {RUN}", "",
    f"Driven {STAMP} against `local`, schema v{OS.schema_version}, "
    f"windows of {WINDOW_SECONDS}s.", "",
    (f"> **WINDOWS.** {WINDOW_PROVENANCE}" if WINDOW_PROVENANCE else ""), "",
    (f"> **PROVENANCE.** {GATE7_PROVENANCE}" if GATE7_PROVENANCE else ""), "",
    f"- **{len(OPS)} operations**, **{len(GRID)} cells**, {len(ROWS)} calls issued",
    f"- covered: {sum(1 for r in ROWS if r['verdict'] == mx.COVERED)}   "
    f"missed: {sum(1 for r in ROWS if r['verdict'] == mx.MISSED)}   "
    f"transport errors: {sum(1 for r in ROWS if r['verdict'] == mx.ERROR)}",
    f"- correlation on error cells (rule 3, an assertion): "
    f"{len(_kept_found)}/{len(_kept)}",
    f"- 500 ladder: {FIVE_HUNDRED['winner'] or 'NOT PROVOKED (expected on this build)'}"
    + f"   |   500s anywhere in the run: {len(ALL_500)}"
    + (f" ({', '.join(sorted({r['op_id'] for r in ALL_500}))})" if ALL_500 else ""),
    "", "## Gates", "",
    _table([{"gate": k, "verdict": {True: "PASS", False: "FAIL", None: "VOID"}[v["ok"]],
             "detail": v["detail"]} for k, v in GATES.items()], ["gate", "verdict", "detail"]),
    "", "## Windows", "",
    _table([{"phase": k, **v} for k, v in sorted(PHASE_WINDOWS.items())],
           ["phase", "start", "seconds", "cells", "straddled"]),
    "", "## Coverage by operation", "",
    _table(COVERAGE, ["op_id", "method", "api", "covered", "missed", "not_driven", "declared"]),
    "", "## Not reachable", "",
    _table(mx.ledger(), ["status", "state", "why"]),
]
pathlib.Path("doc-api-status-matrix-results.md").write_text("\n".join(results), encoding="utf-8")

report = [
    "# For `local-k8s`: what this run says needs changing there", "",
    f"Produced by `log-coverage/polaris_log_coverage_v2.ipynb`, run {RUN}, {STAMP}.",
    "Only items whose fix lives in the pipeline repo are here; everything else stays in",
    "`doc-api-status-matrix-results.md`.", "",
    (f"> **WINDOWS.** {WINDOW_PROVENANCE}" if WINDOW_PROVENANCE else ""), "",
    (f"> **PROVENANCE.** {GATE7_PROVENANCE}" if GATE7_PROVENANCE else ""), "",
    f"**How it was driven:** {len(OPS)} operations from the vendored OpenAPI documents, "
    f"{len(GRID)} (operation x status) cells, boundary-aligned phases so the per-window gates "
    "mean something.", "",
]
for i, f in enumerate([f for f in FINDINGS if f["where"] == "local-k8s"], 1):
    report += [f"## {i}. {f['title']}" + ("  **SERIOUS**" if f["severity"] == "SERIOUS" else ""),
               "", f"**Measured:** {f['measured']}", "", f"**Change:** {f['remedy']}", ""]
# TWO SECTIONS THAT MUST NEVER MERGE (SCENARIO-logging-test.md section 6):
# pipeline findings are local-k8s's work, build findings are Polaris's. This
# report sent Polaris facts to the wrong repo for three runs running.
_build = [f for f in FINDINGS if f["where"] == "polaris"]
if _build:
    report += ["## Build findings -- POLARIS facts, NOT `local-k8s` work", "",
               "Here only because this run provoked them. Nothing in this section is a change "
               "for the pipeline repo.", ""]
    for f in _build:
        report += [f"### {f['title']}" + ("  **SERIOUS**" if f["severity"] == "SERIOUS" else ""),
                   "", f"**Measured:** {f['measured']}", "", f"**Note:** {f['remedy']}", ""]
_guide = [f for f in FINDINGS if f["where"] == "guide"]
if _guide:
    report += ["## For `GUIDE-schema-v3-testing.md` itself", ""]
    for f in _guide:
        report += [f"### {f['title']}", "", f"**Measured:** {f['measured']}", "",
                   f"**Change:** {f['remedy']}", ""]
pathlib.Path("REPORT-for-local-k8s.md").write_text("\n".join(report), encoding="utf-8")

print("\n  wrote doc-api-status-matrix-results.md")
print("  wrote REPORT-for-local-k8s.md")

## 17 -- Teardown

Close the sessions. Nothing here holds a Pgpool-II connection, but the HTTP sessions and the
frame are dropped explicitly so a re-run starts clean.

In [22]:
import gc

for _c in (adm_pc, adm_ic, run_ic, denied_pc):
    try:
        _c.session.close()
    except Exception:
        pass
try:
    OS.session.close()
except Exception:
    pass

del M
gc.collect()
print("  sessions closed, frame released")
print(f"\n  run {RUN} complete -- {len(ROWS)} calls, {len(GATES)} gates, {len(FINDINGS)} findings")

  sessions closed, frame released

  run 1789370776 complete -- 286 calls, 11 gates, 30 findings
